# B08D — Efisiensi Student sebelum dan sesudah Knowledge Distillation

Notebook forward-only untuk checkpoint **B05B/B05C Hinton** dan **B06B/B06C feature KD terinspirasi SSID-KD** yang sudah ditinjau.

- Default: empat dataset, dua Student (MobileNetV2/NASNetMobile), dua pipeline, masing-masing kontrol dan dua Teacher: **48 checkpoint, 96 GPU jobs**.
- **Tidak melakukan training, tuning, atau inferensi ulang pada test.** Dataset citra tidak perlu diekstrak. Metrik test dibaca dari laporan terdahulu yang terikat hash.
- Memeriksa hash checkpoint, restore backbone/classifier, lalu ekspor bobot deployment tanpa projector/Teacher/optimizer. Audit ekspor dilakukan pada CPU; profil dijalankan pada GPU dalam proses terpisah.
- Hasil baru di `outputs/benchmark_4datasets/b08d_post_kd_efficiency_v1/<campaign_id>/`; sumber B05/B06/B08C tidak diubah.

**Cara menjalankan:** gunakan runtime GPU baru **L4**, mulai Cell 1 hingga Cell 8. Untuk mencoba satu dataset dahulu, ubah `DATASETS_TO_RUN=['cbis_ddsm']` di Cell 2. Setelah selesai, pilih dataset berikutnya dan jalankan Cell 2–8 lagi. ID kampanye tidak berubah hanya karena pilihan dataset berubah, sehingga tabel akhir menghimpun pekerjaan selesai dalam kampanye yang sama.

Jika runtime terputus: buka notebook ini di runtime baru, jalankan kembali dari Cell 1. Pekerjaan selesai dan lolos pemeriksaan akan dilewati. Pekerjaan yang sedang berjalan ketika terputus diulang. Perubahan hardware/software/protokol menghasilkan kampanye berbeda. Jangan menjalankan training/profiling lain pada GPU yang sama.

Durasi bergantung terutama pada pembuatan model dan 96 proses GPU; ini bukan training, tetapi NASNetMobile serta audit ekspor tetap dapat memerlukan waktu. Progress dan log disimpan per pekerjaan. Tidak ada estimasi durasi yang dipaksakan sebelum pengukuran.


## Cell 1 — Dependensi dan Drive
Parent notebook tidak mengimpor TensorFlow. Jika sudah menjalankan notebook training dalam session ini, restart session terlebih dahulu.

In [ ]:
import os,sys,json,hashlib,shutil,subprocess,zipfile,uuid,time,select,ast
import importlib.metadata as metadata
from pathlib import Path
from datetime import datetime,timezone
if 'tensorflow' in sys.modules or 'tf_keras' in sys.modules:
    raise RuntimeError('Restart session dahulu. Notebook parent harus bebas TensorFlow/GPU allocations.')
try: TF_VERSION=metadata.version('tensorflow')
except metadata.PackageNotFoundError: raise RuntimeError('Gunakan runtime Colab GPU dengan TensorFlow.')
minor='.'.join(TF_VERSION.split('.')[:2])
try: kv=metadata.version('tf-keras')
except metadata.PackageNotFoundError: kv=''
if '.'.join(kv.split('.')[:2])!=minor:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])
try:import openpyxl
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','openpyxl'])
import pandas as pd
import numpy as np
from google.colab import drive,files
from IPython.display import display
drive.mount('/content/drive')
GPU_INFO=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True).strip()
print('TensorFlow',TF_VERSION,'tf-keras',metadata.version('tf-keras'));print(GPU_INFO)


## Cell 2 — Konfigurasi
Semua pasangan Teacher selalu disertakan. Default pengukuran mengikuti B08C. Jika software berbeda, guard berhenti; jangan mencampur hasil antar-kampanye. `CAMPAIGN_LABEL` baru membuat pengukuran ulang terpisah.

In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASETS_TO_RUN=['cbis_ddsm','cmmd','inbreast','mias']
PIPELINES_TO_RUN=['hinton','feature']
STUDENTS_TO_RUN=['mobilenet_v2','nasnet_mobile']
# All controls and both teachers are always included for a selected student/pipeline.
BATCHES=[1,8]
WARMUP=30
TRIALS=5
ITERATIONS=50
CAMPAIGN_LABEL='main'  # Change only to intentionally repeat every profiling job.
REQUIRE_L4=True
REQUIRE_ORIGINAL_SOFTWARE=True
VERIFY_SAVED_TEST_FILES=True
DOWNLOAD_REPORT=True
JOB_TIMEOUT_SECONDS=1800
if REQUIRE_L4 and 'L4' not in GPU_INFO:raise RuntimeError('Pilih NVIDIA L4 agar sesuai B08C; GPU lain memerlukan kampanye terpisah.')
if REQUIRE_ORIGINAL_SOFTWARE and (TF_VERSION!='2.20.0' or metadata.version('tf-keras')!='2.20.1'):
    raise RuntimeError('B08C memakai TensorFlow 2.20.0 / tf-keras 2.20.1. Gunakan runtime yang sesuai, atau set REQUIRE_ORIGINAL_SOFTWARE=False untuk kampanye berbeda; jangan gabungkan angka lintas environment.')
if not DATASETS_TO_RUN or not set(DATASETS_TO_RUN)<={'cbis_ddsm','cmmd','inbreast','mias'}:raise ValueError('Invalid dataset selection')
if not PIPELINES_TO_RUN or not set(PIPELINES_TO_RUN)<={'hinton','feature'}:raise ValueError('Invalid pipeline selection')
if not STUDENTS_TO_RUN or not set(STUDENTS_TO_RUN)<={'mobilenet_v2','nasnet_mobile'}:raise ValueError('Invalid student selection')
if BATCHES!=[1,8] or WARMUP<1 or TRIALS<2 or ITERATIONS<2:raise ValueError('Use batches [1,8], positive warmup and at least two trials/iterations')
for values in [DATASETS_TO_RUN,PIPELINES_TO_RUN,STUDENTS_TO_RUN]:
    if len(set(values))!=len(values):raise ValueError('Duplicate selections')
print('Selected checkpoints:',len(DATASETS_TO_RUN)*len(PIPELINES_TO_RUN)*len(STUDENTS_TO_RUN)*3)


## Cell 3 — Sumber terkunci, builder, worker, dan fungsi tabel
Cell panjang ini menyimpan manifest 48 checkpoint dan metrik dari delapan ZIP test yang sudah ditinjau. Tidak memilih checkpoint berdasarkan hasil test. Tidak ada fungsi `.fit()` atau pembaruan bobot.

In [ ]:
CATALOG=json.loads('[{"run_id": "cbis_ddsm__hinton__mobilenet_v2__from__densenet121", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.data-00000-of-00001": "de3fba9ee639fd9430c3ad52f770e62de2b213f71290eae0cc15d816c17a2181", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.index": "ad4e05f0ea15b23bffef6f20374855897f5fc6e878e0befe8569311f6d400ebf"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42/config.json": "c3e61f46a9c6511305dff4d62fac5e872ba2ab63fd7255d082e6015c37909ead", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "ea60d54fa43840cae4c318399edc4ecdaff17619376e8d36edd711b48c350d91"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/test_metrics.json": "a4c74443bb1b16d9a40338a98d102ed4a2eed0af0c7abfb299bc1fa100b74a30", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6456310679611651, "recall": 0.6519607843137255, "f1": 0.6487804878048781, "macro_f1": 0.6756299133239266, "accuracy": 0.6778523489932886, "balanced_accuracy": 0.6757746308399903, "specificity": 0.6995884773662552, "roc_auc": 0.7360606794157993, "average_precision": 0.6607835723065891, "tn": 170, "fp": 73, "fn": 71, "tp": 133, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cbis_ddsm__hinton__mobilenet_v2__from__none", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-22", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-22.data-00000-of-00001": "abe104c3ab9f637666e5d30bc3a87dd9c64153955dfb985ecae3b692a5a25eb5", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-22.index": "355f245f1c2278b3142644eff836834bf7b287a6a41d5edab51b12e023bd003b"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/config.json": "be78abbc91b83b8771ee4675fd53610e2401d54c4d806ac742f24ee7e09af696", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/COMPLETE.json": "d0814f9120580cbf3c916f627ae389d4a213707ba691f1350cd3f7f6a4cb20a8"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/test_metrics.json": "6deccef66097b2dc99d10310d8acd03182637ec7d3cf81b66816ce045f0b1709", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6208530805687204, "recall": 0.6421568627450981, "f1": 0.6313253012048192, "macro_f1": 0.6559549261765223, "accuracy": 0.6577181208053692, "balanced_accuracy": 0.6564693778746067, "specificity": 0.6707818930041153, "roc_auc": 0.7048938917130638, "average_precision": 0.6342726525609912, "tn": 163, "fp": 80, "fn": 73, "tp": 131, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cbis_ddsm__hinton__mobilenet_v2__from__vgg16", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-10", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-10.data-00000-of-00001": "a60f0b96ebe12898872b00cb63457b97dbd81c337460fbdcc5d80a2124c9f08a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-10.index": "7ad50471d407325b71513cf893ddb36907d3ad21bd96f5594a572d362e9b0f4e"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42/config.json": "369b104b430264b6ccd0d369773541d98c0810d5f5dd47a98134a21b9adb3a64", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "a3911cfe699b9d0378c22fd06cada8e6e15c4fb72268c725754c867f71c8b63a"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/test_metrics.json": "deca7cf447a0df9d3454a3af33911a4baee06c44edacb24ab4a6bdcae2c84813", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6432748538011696, "recall": 0.5392156862745098, "f1": 0.5866666666666667, "macro_f1": 0.6440077071290944, "accuracy": 0.6532438478747203, "balanced_accuracy": 0.6440934398450738, "specificity": 0.7489711934156379, "roc_auc": 0.741668683934479, "average_precision": 0.66909474634924, "tn": 182, "fp": 61, "fn": 94, "tp": 110, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cbis_ddsm__hinton__nasnet_mobile__from__densenet121", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-17", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-17.data-00000-of-00001": "cebfb0c7936144c17adfa6158db0aa3f9fe5eda3bdcb86513d5069d37afe3ce1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-17.index": "a9867cf41219d95846bf66b767fa79a7dff3fe2ceefb7bd1d31cd422bdbe7aca"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42/config.json": "1feb6866944565806e535e580ecd70427d5b8649d319e3285eb01a4c5e8fd3b6", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "122db22ad97f23387ea688d9accc1026b6d5c24cc238a0707877f664c994da48"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/test_metrics.json": "19449d7f941c907fe9b3ac04ca0c134944b6e0b0d9662e608d7dfb81743a4bb2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6355140186915887, "recall": 0.6666666666666666, "f1": 0.6507177033492823, "macro_f1": 0.671997507136826, "accuracy": 0.6733780760626398, "balanced_accuracy": 0.6728395061728395, "specificity": 0.6790123456790124, "roc_auc": 0.7518558863874768, "average_precision": 0.7056954710275896, "tn": 165, "fp": 78, "fn": 68, "tp": 136, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cbis_ddsm__hinton__nasnet_mobile__from__none", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-25", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-25.data-00000-of-00001": "3c66146509032b0d65b1254f13d0df4da11f28dcf29e7996fd0d769c63d99d3b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-25.index": "6d3b4d6d33d8873ec40db913ee8634953d8f4b1fab4ea7e0d0d4f7dfe1a1d659"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/config.json": "d10bd66bc3bb119a474fca1a901c6ae5b77e037ed5a3a6f0ec57f543b86ad88d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/COMPLETE.json": "7da10275f7b2edfb060d1f3c37dfece1e65af19d5c9d26c036e72c3c56569912"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/test_metrics.json": "65c60fcdd186c70c83652c03be1a7a2ad0d2e6b94d9857b55d6b0082d0719d4e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6507936507936508, "recall": 0.6029411764705882, "f1": 0.6259541984732825, "macro_f1": 0.6662705124102939, "accuracy": 0.6711409395973155, "balanced_accuracy": 0.6656681190994916, "specificity": 0.7283950617283951, "roc_auc": 0.7237351730815783, "average_precision": 0.6682067288670491, "tn": 177, "fp": 66, "fn": 81, "tp": 123, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cbis_ddsm__hinton__nasnet_mobile__from__vgg16", "dataset": "cbis_ddsm", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13.data-00000-of-00001": "39fefdb2d8c436ff130e6fa4bf3007e71a938c2294caffb184c3f3f53df0b01b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13.index": "93a9fcbfd4727e0eb89513fea1d52d5956e50bc73f4075a7fc0b5dd62d9553b1"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/protocol.json": "070a5534a2a884c985d45f05d1eef4547905b253f8ab96b9d066ddec7f4efc2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42/config.json": "83681206a0b24a9cca1802c00d0f0a23629d61eff164ee3120cb1409097fa51c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "fdfa5b263a954e3bfcf3bfe53006c2bf9574510f98480d2c87a6aae7ca523f70"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/test_metrics.json": "172dab9db601ba0311376bac10f76cafaf523395fb7213f5f2fa684e85f918a4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cbis_ddsm/7a4bf6d1e3f6/evaluation_lock.json": "353cc2a022e0b9a00e291997c90f5c833222936f96d881b47548cb5f1459016d"}, "metrics": {"precision": 0.6625, "recall": 0.5196078431372549, "f1": 0.5824175824175825, "macro_f1": 0.6478125647936969, "accuracy": 0.6599552572706935, "balanced_accuracy": 0.6486928104575164, "specificity": 0.7777777777777778, "roc_auc": 0.7459251190188009, "average_precision": 0.7285925466668637, "tn": 189, "fp": 54, "fn": 98, "tp": 106, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cbis_ddsm_seed_42_7a4bf6d1e3f6_report.zip", "report_sha256": "a4bcd0fdac00dbaa9b2b6979ff9691cb5a3dd7c713246e96eb7514fc35be71bc"}, {"run_id": "cmmd__hinton__mobilenet_v2__from__densenet121", "dataset": "cmmd", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-8", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-8.data-00000-of-00001": "dece6f29fbe42040439f20a0c30852cd01d3e20b949f635c7b285412a792e5f8", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-8.index": "a1f2839b34d954636fe7efed90d1dfc69f47a74578898cc8dd7bf98c0d14f53a"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42/config.json": "276040d2b493e22d90b91e0a559e1a60acc4381b8c6703a069917d10b22fb3a5", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "6fb61a7ab85d210c5cd6d5607312a5ba829129227e3e57a61dee3359bd8d14a3"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/test_metrics.json": "e0c678e0b4c4116b493326d3e989ef6d3c1f1e87d4993ed463b816a006aaae8c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.8257142857142857, "recall": 0.733502538071066, "f1": 0.7768817204301075, "macro_f1": 0.6850094876660342, "accuracy": 0.7118055555555556, "balanced_accuracy": 0.6991688514531154, "specificity": 0.6648351648351648, "roc_auc": 0.761044792770681, "average_precision": 0.8873080960316059, "tn": 121, "fp": 61, "fn": 105, "tp": 289, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "cmmd__hinton__mobilenet_v2__from__none", "dataset": "cmmd", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-16", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-16.data-00000-of-00001": "2be8bb9004bd5520f2e57bd2c6333fca2c574e278c31cf73033ad2e3fb918a76", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-16.index": "0387da1d7d3258b495428657b4105e4551025e120edd77edb4bd3207d487f269"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42/config.json": "4a7e5706dd0b2b73a4ff54e95dd2473cf9ade90c25f2459641ed3a166b22cf8a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42/COMPLETE.json": "e2f36bebcf8fba591e24edaf2d6523c7596783b3957f99a0e589c724013db505"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/test_metrics.json": "246d9eb7141522da237ca7ce16d3390cc34dc741af604044d1d387cdf2d1c6f8", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.7818181818181819, "recall": 0.7639593908629442, "f1": 0.772785622593068, "macro_f1": 0.6491273957469361, "accuracy": 0.6927083333333334, "balanced_accuracy": 0.6512104646622413, "specificity": 0.5384615384615384, "roc_auc": 0.7424276231382831, "average_precision": 0.8722796907320898, "tn": 98, "fp": 84, "fn": 93, "tp": 301, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "cmmd__hinton__mobilenet_v2__from__vgg16", "dataset": "cmmd", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-5", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-5.data-00000-of-00001": "67cc678cf59e9122085e52148088a4280f19300388a99234eec3b7f9b778f2d3", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-5.index": "3727cf1a2012ad74d676073d589719ba95c4101b32533f9cac7ac3f46b73600d"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42/config.json": "149d04af1e9d565db7a4bed12720c7e3b85d22063499204cc9b5673dee37a50c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "a1e12224c13cffe038b873916095510e0fd491474fe1cc1ae8bef8d1cc1b4464"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/test_metrics.json": "37aa4fb6baac3f28ce7a951b4fa8d2161a59ad3f80b08b0fd742e1fe304d837e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.8096590909090909, "recall": 0.7233502538071066, "f1": 0.7640750670241286, "macro_f1": 0.6652887650391579, "accuracy": 0.6944444444444444, "balanced_accuracy": 0.6776091928376192, "specificity": 0.6318681318681318, "roc_auc": 0.7415351146315612, "average_precision": 0.8730434229319248, "tn": 115, "fp": 67, "fn": 109, "tp": 285, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "cmmd__hinton__nasnet_mobile__from__densenet121", "dataset": "cmmd", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23.data-00000-of-00001": "9325c368c6cd5a156a352359acf49baad2c1909876794f98a21a281368f038a8", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23.index": "a6dffd6788997caf92fcb343ffd9816a073ac2e038cd49e1e9e4e1e28e3b4e81"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42/config.json": "e337ab83ec54f5e66da40ac05d7d7610b184c91b28ff8cce9b75d1859386c723", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "408136c90935d36474de5080b470c1b22df23280b22125b0b58dd3b411951b1d"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/test_metrics.json": "a4514e5c5fc058c948364d464436607d56e9a32593d360dfa7cbcebca8a10153", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.7950819672131147, "recall": 0.7385786802030457, "f1": 0.7657894736842106, "macro_f1": 0.6558539205155747, "accuracy": 0.6909722222222222, "balanced_accuracy": 0.6632453840575668, "specificity": 0.5879120879120879, "roc_auc": 0.7413817147319686, "average_precision": 0.8709674935767301, "tn": 107, "fp": 75, "fn": 103, "tp": 291, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "cmmd__hinton__nasnet_mobile__from__none", "dataset": "cmmd", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-24", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-24.data-00000-of-00001": "35478fb9c64b72c9ab554e959b7931ae45a7cb4bf713a0a1592f11f3f4883e33", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-24.index": "1de15fda7da50a9207081c89c3e1be1b9d76b83fc37c883d82409445432e6888"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42/config.json": "14e1e4fcc23952abb9a94b6c1d9aee8341906b573dc0746846e9a4e25ad9a51d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42/COMPLETE.json": "d22c40dcf6aa9b8bf8d2069341cc39caf845ea017d61794513f569218fd7d0a0"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/test_metrics.json": "5566c3c40602e82109d7b46745200b30b2b2c71765fc47f98a6ed7d6b68c0f5c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.7976539589442815, "recall": 0.6903553299492385, "f1": 0.7401360544217687, "macro_f1": 0.641051240640141, "accuracy": 0.6684027777777778, "balanced_accuracy": 0.6556172254141797, "specificity": 0.6208791208791209, "roc_auc": 0.7040218664584147, "average_precision": 0.844156742888278, "tn": 113, "fp": 69, "fn": 122, "tp": 272, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "cmmd__hinton__nasnet_mobile__from__vgg16", "dataset": "cmmd", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14.data-00000-of-00001": "6748e60cefd11cec46d70c32be157baf656d82839170948605839427fd1da252", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14.index": "d2e4b86c4f87b4248175dd36de79e8708e896a854565c64dcdbab39927f97070"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/protocol.json": "eb3102fdcfeee6abac3510cce02f95be137883ed0e02704a5d545a3e04a8b5bb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42/config.json": "42d6d44665a69eac1fff4deb449bf8990df217d92ce63cf79ef39a489761cecf", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "53bc3e455465179ccfd74aa55b0dcb7a2781f74161b76b20e452f624fdd55236"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/test_metrics.json": "720fbd9700a164a824e382ed18baa2803f579d9bbb79f801e7ae40e5ccef9e64", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/cmmd/f81ede1d807a/evaluation_lock.json": "413063b3c078fb1ac87ce33ca20be8b14c86c2abd52adf5475ed27f6e1d04060"}, "metrics": {"precision": 0.7774936061381074, "recall": 0.7715736040609137, "f1": 0.7745222929936306, "macro_f1": 0.646116732327878, "accuracy": 0.6927083333333334, "balanced_accuracy": 0.6467758130194678, "specificity": 0.521978021978022, "roc_auc": 0.7427065320466335, "average_precision": 0.8669609149463977, "tn": 95, "fp": 87, "fn": 90, "tp": 304, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "dataset": "cmmd", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cmmd/f81ede1d807a/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_cmmd_seed_42_f81ede1d807a_report.zip", "report_sha256": "133f509df8dca950de9ab71b5494a1a33f9cd865c07acfbd1c5d2b51caabb627"}, {"run_id": "inbreast__hinton__mobilenet_v2__from__densenet121", "dataset": "inbreast", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.data-00000-of-00001": "d4c5651ab081fbabfc2086d1a212da952d783e64c1a544e90b3e65ef7c7ce684", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.index": "782c55f57b8e2f2ea1010ddfe9fe221f7358db487436a3220a241cf7bd6dc4ce"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42/config.json": "33a4acce90b9a26a794e9760a4f4576327fa8a2fd438234ac329c65d9d0c6efb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "1a5a7d832ad41fcb9a3ea483d65d63d4e62bceba2388f3fa9eac816938aeb88d"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/test_metrics.json": "82f34ab7fb53cb354d2f11ea442027db9b0cf9f3930434200d31f628ebae3cf1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.5, "recall": 0.7142857142857143, "f1": 0.5882352941176471, "macro_f1": 0.756483238456673, "accuracy": 0.8727272727272727, "balanced_accuracy": 0.8050595238095238, "specificity": 0.8958333333333334, "roc_auc": 0.8333333333333333, "average_precision": 0.3839587447882379, "tn": 43, "fp": 5, "fn": 2, "tp": 5, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "inbreast__hinton__mobilenet_v2__from__none", "dataset": "inbreast", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-14", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-14.data-00000-of-00001": "1063ec9c4ccfde85538a41d2656658bd564254ef1c98bab205f7a05fc73473a7", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-14.index": "c2bc2ad8ff0015777704193f374c77d6061dc8d240f3e531c44157bad6988dde"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42/config.json": "7e1405207822ee7a8ccb142353f5348c84f255b579f0f357f4fe5ad8b397c52b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42/COMPLETE.json": "4803b70fd243e7ad75c7405a31f2836726b44aaf3b60b7f35bb77e2b5905f1fe"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/test_metrics.json": "24f0295096de73cc64b21fca1146477db888c0c05bd07a646b84754a0f2c00e6", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.3333333333333333, "recall": 0.42857142857142855, "f1": 0.375, "macro_f1": 0.6343085106382979, "accuracy": 0.8181818181818182, "balanced_accuracy": 0.6517857142857143, "specificity": 0.875, "roc_auc": 0.7410714285714285, "average_precision": 0.29179748822605966, "tn": 42, "fp": 6, "fn": 4, "tp": 3, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "inbreast__hinton__mobilenet_v2__from__vgg16", "dataset": "inbreast", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25.data-00000-of-00001": "344dee78a15262fe5f99b8d1bb152a5161d5f4fe45fd1525302ecb4b10d9e29c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25.index": "5dc9c818cb33447092a992fa35931c2a77ba9e9de5d4f38855884c91b460bc72"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42/config.json": "0204dd2df6f9e3f6e04c249612a62403c9b2236fc74af6b62107c257458c6e3f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "bb5e2c1cae237f53e7b72d4dbe362c59c8c871f535331c2e4f8a6e7e5d804160"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/test_metrics.json": "c8a2d52e93108b15375edf74ad9fa1aeef43ecdd2f5b9fbe17c9e89b23826501", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.625, "recall": 0.7142857142857143, "f1": 0.6666666666666666, "macro_f1": 0.8070175438596491, "accuracy": 0.9090909090909091, "balanced_accuracy": 0.8258928571428572, "specificity": 0.9375, "roc_auc": 0.8303571428571428, "average_precision": 0.4435236256664828, "tn": 45, "fp": 3, "fn": 2, "tp": 5, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "inbreast__hinton__nasnet_mobile__from__densenet121", "dataset": "inbreast", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-16", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-16.data-00000-of-00001": "7e9d4f16d2882ba4dc2eda18c34e8ac49afc7643287f23e32cf647b5ffc72972", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-16.index": "de167ffc3f1eb644070cff3f83752ba8ce91c922064cc5dd7a00fba60161c700"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42/config.json": "aa89b87f2d709f6bbb1f9035d188aff029bb727e85d8245eb2834c9567747472", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "bb6c14f9724022b975251d61947d19b55ae2f8f8729bf1b8cb4056429bb18aea"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/test_metrics.json": "0cdb8c8a78671a8c0717534589d3d3ce512946e9db47caac56edbe8b0c608277", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.25, "recall": 0.14285714285714285, "f1": 0.18181818181818182, "macro_f1": 0.5454545454545454, "accuracy": 0.8363636363636363, "balanced_accuracy": 0.5401785714285714, "specificity": 0.9375, "roc_auc": 0.5803571428571429, "average_precision": 0.22100454526925115, "tn": 45, "fp": 3, "fn": 6, "tp": 1, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "inbreast__hinton__nasnet_mobile__from__none", "dataset": "inbreast", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-20", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-20.data-00000-of-00001": "66079d014d1b0c90ca61d833d1a622bf89c810a12651a19cc41a1898011c406d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-20.index": "0827da22676918e82c56082839bbaf955f8e33015a744595f46eaf86367898aa"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42/config.json": "97de7d12f92c21c917881e76afcae800389689614e19f8a82328602050a6a9db", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42/COMPLETE.json": "4a493989ea0f53ad44868f79b523509a9c5f7761d48e1c304e61f292f1f5a0a0"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/test_metrics.json": "8b2681f9bb4befdca43548cea3c8afafd6006282d0204fcf481806b225c8a16d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.5, "recall": 0.2857142857142857, "f1": 0.36363636363636365, "macro_f1": 0.6464646464646464, "accuracy": 0.8727272727272727, "balanced_accuracy": 0.6220238095238095, "specificity": 0.9583333333333334, "roc_auc": 0.6904761904761905, "average_precision": 0.4273916463136979, "tn": 46, "fp": 2, "fn": 5, "tp": 2, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "inbreast__hinton__nasnet_mobile__from__vgg16", "dataset": "inbreast", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14.data-00000-of-00001": "d2c9758fec39658f423e728e60b0cbf07a44f4749303de9a5fdd1c456f1b6459", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-14.index": "144f94c0eacf19fea7a26f47193288e7c2f31e8c8bc3d104592581b45a3c99ac"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/protocol.json": "f9ebf752f1445afc62506918149b9412cdd47d85682b8998a651dcd842e40031", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42/config.json": "b199d8452ee432cae5d4199c4e97ce6427aebebe45ff95a13cb4289d26ad7120", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "85b47171c652b4eefe63ba4cc5e57969f404b64ccebd6fbb0eee2c001e6af816"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/test_metrics.json": "2be0d0eb05e661037fd4e0d207a307dc4a6cedc391c45e6aef481380574afaa0", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/inbreast/3f2fbdc2b913/evaluation_lock.json": "a4e34b33f1b164d52b1fb180df0037fd045cfd804f15407523a2bf18da06b2e6"}, "metrics": {"precision": 0.3333333333333333, "recall": 0.5714285714285714, "f1": 0.42105263157894735, "macro_f1": 0.6500867553499132, "accuracy": 0.8, "balanced_accuracy": 0.7023809523809523, "specificity": 0.8333333333333334, "roc_auc": 0.6398809523809523, "average_precision": 0.3010539678748772, "tn": 40, "fp": 8, "fn": 3, "tp": 4, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "dataset": "inbreast", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/inbreast/3f2fbdc2b913/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_inbreast_seed_42_3f2fbdc2b913_report.zip", "report_sha256": "a45e848a91225e26972ba1330dea64664226395c9eb6c3170fb8ed065b368269"}, {"run_id": "mias__hinton__mobilenet_v2__from__densenet121", "dataset": "mias", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-24", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-24.data-00000-of-00001": "831e14a92819943b819cd891dc965e3b3d62e6db3d6bb190400e4ee0e7e4aa34", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-24.index": "4afeb0c8861fc9129e448796cb9eb904313cf1e3e14c1a0217a5319ec8ac88b9"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42/config.json": "0b145e49ca3d930b80c447cc55621063d48ccba63aebb387146e910f7982fcb9", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "1dafd12838b88e9269e1b50f562a77d65fbfe0edcc9d6176cadf275bc8427e8a"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/test_metrics.json": "4ed1f42766e467dd41eacd06716d634b487f36466b8a1b65f21c715a2dd70d92", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.46153846153846156, "recall": 0.75, "f1": 0.5714285714285714, "macro_f1": 0.43956043956043955, "accuracy": 0.47058823529411764, "balanced_accuracy": 0.4861111111111111, "specificity": 0.2222222222222222, "roc_auc": 0.4444444444444444, "average_precision": 0.49759222599296127, "tn": 2, "fp": 7, "fn": 2, "tp": 6, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "mias__hinton__mobilenet_v2__from__none", "dataset": "mias", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-13", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-13.data-00000-of-00001": "9ac4f0d64a7aa21f6343ee05f81c7513a39f714ce4e1ea74a151862b1f438c68", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-13.index": "df24683eb336db804c119b1e17ba4cc3181a2409b1b2cb3be48a26a01cbf484c"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42/config.json": "c4392e0b905afa3d7ff3c4cc31f1f7a8a7e55e548af48714853144df85e6e07e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42/COMPLETE.json": "bd5e6eed559764025234640da0aaeefc8baa0cf88deb75c7edf8503e9a6e487c"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/mobilenet_v2__from__none/test_metrics.json": "060d1b93c069bca5e2b6952df1dcee5607179408bf6e36309ce6f7f23c68352e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.45454545454545453, "recall": 0.625, "f1": 0.5263157894736842, "macro_f1": 0.4631578947368421, "accuracy": 0.47058823529411764, "balanced_accuracy": 0.47916666666666663, "specificity": 0.3333333333333333, "roc_auc": 0.3888888888888889, "average_precision": 0.45319356909430436, "tn": 3, "fp": 6, "fn": 3, "tp": 5, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "mias__hinton__mobilenet_v2__from__vgg16", "dataset": "mias", "pipeline": "hinton", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-7", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-7.data-00000-of-00001": "bc11a7afcab2de60291e10c410219f9b2064a0b398d15b0e3a6954156a64749f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-7.index": "f253ffd1038590b3edcb33b5231362db5394ca9360beb6f0127f06458961d142"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42/config.json": "d8d8e3ac9dead406a37d0afa0a45542b1f30647721a2c7f8f7e85e83f94283cb", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "f1751347dfe454d7506ad040a763e6b330c0a8155059b585f36780d50cd87693"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/test_metrics.json": "c38794566f9793989771d82e3a11bda2ef90128fcdb037cf3b1f5c27c60ae0e8", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.16666666666666666, "recall": 0.125, "f1": 0.14285714285714285, "macro_f1": 0.27142857142857146, "accuracy": 0.29411764705882354, "balanced_accuracy": 0.2847222222222222, "specificity": 0.4444444444444444, "roc_auc": 0.23611111111111113, "average_precision": 0.36754033344106873, "tn": 4, "fp": 5, "fn": 7, "tp": 1, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "mias__hinton__nasnet_mobile__from__densenet121", "dataset": "mias", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42/head/best/ckpt-2", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42/head/best/ckpt-2.data-00000-of-00001": "05b3ea0c180636b3f35f1243d1c4e5f52aa382715da73dfe4c25528a93105186", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42/head/best/ckpt-2.index": "23a3a21f778cc4a3e840093e6d1ea78b45ae412b68f2578a505a865fa26becbb"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42/config.json": "6bff25a287bd4023270c0f6eb9ac094c366dce8c6bfc9c6364ab37596ca84b43", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "dca08baa3df2e64ed2234e213ae6475c8207cd04e08b0c39f020fd6d27c2f650"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/test_metrics.json": "9183dd367c0627a6983da754c3287d308ef42abae8247e5131fa77718a5bc3b4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.2, "recall": 0.125, "f1": 0.15384615384615385, "macro_f1": 0.315018315018315, "accuracy": 0.35294117647058826, "balanced_accuracy": 0.3402777777777778, "specificity": 0.5555555555555556, "roc_auc": 0.4583333333333333, "average_precision": 0.45488233172056697, "tn": 5, "fp": 4, "fn": 7, "tp": 1, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "densenet121", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "mias__hinton__nasnet_mobile__from__none", "dataset": "mias", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42/head/best/ckpt-4", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42/head/best/ckpt-4.data-00000-of-00001": "f3229a27c7edfae563c6c616724eb0ac8c060748488a365c92d395756272dd11", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42/head/best/ckpt-4.index": "e7c76f3d723a9bc7e25060f5cf2dcda8e73b2cbfd6d72d9fcc6c2d7e24aaf08c"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42/config.json": "f22bc3db8cfe7e754535fe562c0491a5aac64d410bf5c63a084f0e60ef0fc486", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42/COMPLETE.json": "76b28ea2b7b51b65d21e8ccc2c6f4133e91cf7dc65878c8535397e5378c990d9"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/nasnet_mobile__from__none/test_metrics.json": "1308ea34372c0a51f334407ed63feed2d721dce646c05988bb8d9cf5540a4f2d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.3333333333333333, "recall": 0.25, "f1": 0.2857142857142857, "macro_f1": 0.39285714285714285, "accuracy": 0.4117647058823529, "balanced_accuracy": 0.4027777777777778, "specificity": 0.5555555555555556, "roc_auc": 0.4166666666666667, "average_precision": 0.434889763014763, "tn": 5, "fp": 4, "fn": 6, "tp": 2, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "none", "method": "supervised_control", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "mias__hinton__nasnet_mobile__from__vgg16", "dataset": "mias", "pipeline": "hinton", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-21", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-21.data-00000-of-00001": "ea12efd295a6e2ec298573b1a6f08acdae53bf67f9f9f2c1292fc3cf42b5c245", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-21.index": "7fdbebad011de14b92b33efaa19e53bf3f4105878b4de4c26a966ea1632b1225"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/protocol.json": "be9d7ccecaf36dee0d6f2acc1530d207ec75aadeba795c37b5e6a5ad4c69b64b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42/config.json": "4ef235662182480425ff8111e755cd029560bbcc75c75965bf267b61cd50c36c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "af7f06e34aa989ed2a29d2aee672550c387d918ae91c33166c10d88fc20dcbc4"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/test_metrics.json": "433eebb1391225dd6c98e050af1161b4644ebb64008fc97e94ede96bbc6339c2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05c_fixed25_test_v1/mias/672f572d6c33/evaluation_lock.json": "4a1624e55c2bf1ff922ce3ad184aa10b462a394fabdac5fc8b15a5f7c2340851"}, "metrics": {"precision": 0.16666666666666666, "recall": 0.125, "f1": 0.14285714285714285, "macro_f1": 0.27142857142857146, "accuracy": 0.29411764705882354, "balanced_accuracy": 0.2847222222222222, "specificity": 0.4444444444444444, "roc_auc": 0.3611111111111111, "average_precision": 0.48868845615169143, "tn": 4, "fp": 5, "fn": 7, "tp": 1, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "dataset": "mias", "training_seed": 42, "split": "test", "role": "student", "teacher": "vgg16", "method": "hinton_kd", "run_folder": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/mias/672f572d6c33/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"model": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 0.0, "class_disagreements": 0}, "report": "B05C_Fixed25_mias_seed_42_672f572d6c33_report.zip", "report_sha256": "70234f4c763c46e4e0c31208c716f3a98de556f05996874d93f2e94ca4a66c2f"}, {"run_id": "cbis_ddsm__feature__mobilenet_v2__from__densenet121", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-23", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-23.data-00000-of-00001": "30c976be700987445a3705c1210608c914c073ad7085645f7a5adaa4bf0bba1a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-23.index": "b6aa228f17a27cfb8bbc268c8750b73aa71979164eb831dbc66cbf6b24116cfc"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42/config.json": "a7ae1e4a5dc4bbe525374a1c1f83b6af53158cf0b95722bc95d620be772bdd88", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "53141539528d9d97ef26b3207453adb629c4bba710004dfd2d22c64ee21a58ff", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/mobilenet_v2__from__densenet121/test_metrics.json": "f139b707e9c820fa6f7ca567ae6f3d60cdf97f03a8a80b14c509233bef6bc744", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6417910447761194, "recall": 0.6323529411764706, "f1": 0.6370370370370371, "macro_f1": 0.6682117700522608, "accuracy": 0.6711409395973155, "balanced_accuracy": 0.6680283224400871, "specificity": 0.7037037037037037, "roc_auc": 0.7413459210844832, "average_precision": 0.6787262015394775, "tn": 171, "fp": 72, "fn": 75, "tp": 129, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "teacher": "densenet121", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 2.1559554314561082, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.972869872497341e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cbis_ddsm__feature__mobilenet_v2__from__none", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-18", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-18.data-00000-of-00001": "f455d937f09bc35721e1613b3c0d6293b794d1703eb10136349c6bdd584cb7f1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-18.index": "85169eaea9070faaf46c36a6aaca3eda0ad709d2f45a24af747fda5a0cfcdcd8"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42/config.json": "846e6f14a3971383a95b60f8ffd0665a09c394914d4c43ba2043b359e718853f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42/COMPLETE.json": "8a8e2438becc486a8247b87c61c5ae471db26a556808c139dd6a04e5aa1b17f4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/mobilenet_v2__from__none/test_metrics.json": "42ff471dbf57a23a85da81693b2f39f421661fd0bbcde9674a04eef570226e96", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6384976525821596, "recall": 0.6666666666666666, "f1": 0.6522781774580336, "macro_f1": 0.6741474744732516, "accuracy": 0.6756152125279642, "balanced_accuracy": 0.6748971193415638, "specificity": 0.6831275720164609, "roc_auc": 0.7156055837973049, "average_precision": 0.6521860388053002, "tn": 166, "fp": 77, "fn": 68, "tp": 136, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "teacher": "none", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 2.9740142792888946e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cbis_ddsm__feature__mobilenet_v2__from__vgg16", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-23", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-23.data-00000-of-00001": "6677b060ac02e3e8c702a43e5729f874a8e2d0ba1a47d054512494443b7ba70d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-23.index": "ce770cb8b4338f1965304edb6ef1e4870db6802a5ad6f0c808831766d55e32ac"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42/config.json": "1b7aa521cf538a1813be292d6d149f65d1c16f2a968c5ea14c05dfccba8f45e6", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "f3570b00cf400922b9397360351d0513bde249bdde4c944784f488f355ce4655", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/mobilenet_v2__from__vgg16/test_metrics.json": "0d6c80f837e72e334d1fdf171c003552fddfb96e8a4a81e52fc52f49ac44a0c1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6481481481481481, "recall": 0.6862745098039216, "f1": 0.6666666666666666, "macro_f1": 0.6856540084388185, "accuracy": 0.6868008948545862, "balanced_accuracy": 0.6867586540789155, "specificity": 0.6872427983539094, "roc_auc": 0.7343056564189462, "average_precision": 0.6641505934876655, "tn": 167, "fp": 76, "fn": 64, "tp": 140, "threshold": 0.5, "images": 447, "model": "mobilenet_v2", "teacher": "vgg16", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.13254954700560206, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.964477541222976e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cbis_ddsm__feature__nasnet_mobile__from__densenet121", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-8", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-8.data-00000-of-00001": "98a254be454d6d388f05b817319b8f558e0de3968e6155194b0ec5a56d45c4a4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-8.index": "28f1e0c5ad6d0c7157d76fa47bd895bc04841fd82eae7abf33f8f98cd366a1ec"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42/config.json": "6d9d750cb753f9d23b63d6904b99bccc5bbc258d2707b3213c3759a468bb181c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "9c753b24d4b67dc8e5e1e1d5c928a1b03732e680b3516bcb3a394c4eb7bcbd01", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/nasnet_mobile__from__densenet121/test_metrics.json": "19032870ba73f0e81141ac21c9a1a9359d9199ee1589d834c40bf2af54abb9a1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6479591836734694, "recall": 0.6225490196078431, "f1": 0.635, "macro_f1": 0.6697267206477733, "accuracy": 0.6733780760626398, "balanced_accuracy": 0.6692992011619463, "specificity": 0.7160493827160493, "roc_auc": 0.7494755103687566, "average_precision": 0.6896611611705034, "tn": 174, "fp": 69, "fn": 77, "tp": 127, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "teacher": "densenet121", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 1.8621723330739, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.9788208011360954e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cbis_ddsm__feature__nasnet_mobile__from__none", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-8", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-8.data-00000-of-00001": "d5d7698bef683235eb9ee82ecae0ac2fee68c9ee7051f3448dd6f8ec1c675ca3", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-8.index": "e76f37f3d805bce4be8de959565ba6fb22e3eaebfda921f7313db78a7443ef52"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42/config.json": "01753709d0c91cd76530a21d989576c408ce30c8156d1ea278ceb4ce123fdfe9", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42/COMPLETE.json": "d95d943f72162f0d90e09274da4c28d566fc227cbae5fe79cc2cc780aca099a3", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/nasnet_mobile__from__none/test_metrics.json": "b46cf5698a4c1007f6b90f162a9c9a6d87e58a7f2d6ec35a81ed73a2bbd207e1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6138613861386139, "recall": 0.6078431372549019, "f1": 0.6108374384236454, "macro_f1": 0.6435334733101834, "accuracy": 0.6465324384787472, "balanced_accuracy": 0.6434277414669571, "specificity": 0.6790123456790124, "roc_auc": 0.7465706447187929, "average_precision": 0.7172994046714932, "tn": 165, "fp": 78, "fn": 80, "tp": 124, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "teacher": "none", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 2.9534149126320134e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cbis_ddsm__feature__nasnet_mobile__from__vgg16", "dataset": "cbis_ddsm", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-12", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-12.data-00000-of-00001": "b69259829315c34cfa9c99163a991c419d9bc97a0f3feb73dd8c3974fa7df52e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-12.index": "e3cabe33d8ada07cc02b418270fb2069374e353773213722c32e76208ed191d9"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/protocol.json": "0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42/config.json": "7469d9adf43d73cf136ec531f809ee38171b06dfee5740a95bdcfe745180c894", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "f6362ba596085033c517102cdd79d3eb2b7f27d3042e1aaaf32328b024fdff9d", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/nasnet_mobile__from__vgg16/test_metrics.json": "dc7c8bac09cb4cbe132f97b6a011e9db9eab69e54c9bba90734197959f7f0b0c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cbis_ddsm/evaluation_lock.json": "48024cc076e2706c669bf7103cee96e44c359c517bab840afdd5ddd2b0acf874"}, "metrics": {"precision": 0.6607142857142857, "recall": 0.5441176470588235, "f1": 0.5967741935483871, "macro_f1": 0.6547089358546534, "accuracy": 0.6644295302013423, "balanced_accuracy": 0.6547748729121278, "specificity": 0.7654320987654321, "roc_auc": 0.7368272411845397, "average_precision": 0.6982714822154857, "tn": 186, "fp": 57, "fn": 93, "tp": 111, "threshold": 0.5, "images": 447, "model": "nasnet_mobile", "teacher": "vgg16", "dataset": "cbis_ddsm", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.09108524416056783, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cbis_ddsm/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.974395751920156e-08, "class_disagreements": 0}, "report": "B06C_cbis_ddsm_seed42_test_report.zip", "report_sha256": "39684fbc9057193bc49874871d1ac69f8d1fdcbc9741e93b7566b7736a3f72c9"}, {"run_id": "cmmd__feature__mobilenet_v2__from__densenet121", "dataset": "cmmd", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-21", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-21.data-00000-of-00001": "674daf0d7f8730222afc13e3feae1668380b5cd82fd5784e0cd310b02a49afc3", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-21.index": "8d096f1ee5bb8d428b948226d70588a27d20505ac9edd43cb2ccc368b4c79439"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42/config.json": "84d5ea3ce34c1cd57fede737dcab15eca154d3e513265bdff78b3ebed7036c9a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "c3c9db59e25b40b54fbc86b1aa614975f2279a56c840af833bf596784f3e14e8", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/mobilenet_v2__from__densenet121/test_metrics.json": "7006c1f012cda0abb598c9dfd6f1a833c7cc6eab3982374126abee106ca79a20", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.8088642659279779, "recall": 0.7411167512690355, "f1": 0.7735099337748345, "macro_f1": 0.6713897275927069, "accuracy": 0.703125, "balanced_accuracy": 0.6809979360740782, "specificity": 0.6208791208791209, "roc_auc": 0.7506972722708762, "average_precision": 0.877636305869919, "tn": 113, "fp": 69, "fn": 102, "tp": 292, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "teacher": "densenet121", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.06539590871476625, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.977600099818289e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "cmmd__feature__mobilenet_v2__from__none", "dataset": "cmmd", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-12", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-12.data-00000-of-00001": "1ce587c72a5376c5756a53db45dc36d742edc279ae4f395fb86fabc635a3a939", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-12.index": "8cb9b511f26b8e95c6aff49f3bb15e84e3a4af0eed72a866e786d73cbe87ed9f"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42/config.json": "0025a1bee6e8cd0cbfd41d09fbb03cead63eb311b52fb223654e59fd3e62adb5", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42/COMPLETE.json": "74d9ad8f85b27faaa5807429a4f2d4ca72c3acac8fd44abfad828ed8321255dd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/mobilenet_v2__from__none/test_metrics.json": "5aa6a58772ef30e429822b0c5fea6abcae26b8e437cbab2a50f3c57db7e5e039", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.7919799498746867, "recall": 0.8020304568527918, "f1": 0.7969735182849937, "macro_f1": 0.6742527758555887, "accuracy": 0.7204861111111112, "balanced_accuracy": 0.6729932504044178, "specificity": 0.5439560439560439, "roc_auc": 0.7551598148044849, "average_precision": 0.879740834008173, "tn": 99, "fp": 83, "fn": 78, "tp": 316, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "teacher": "none", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 2.971954349284545e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "cmmd__feature__mobilenet_v2__from__vgg16", "dataset": "cmmd", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25.data-00000-of-00001": "5d72d5cea1c8a89884d6f0425c956a95f72223fc4bbc3a4ecae3eb70d1f3101b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-25.index": "4f2d39f6c304d82bda9a3c3aae6ddc304b24986ca21129fde140a535962c9dbe"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42/config.json": "670f81e29a287b8192fe2386dcf32088c63facfd4e7cc518c02170997c6e9f68", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "a65f9476970326e198b95381af507b93a1ff43470fd2e207cb0f0f11b5468694", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/mobilenet_v2__from__vgg16/test_metrics.json": "56094050f9f7106ee2ab3953c1657866ae6c06a15f6c25d179cc590b83da82d6", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.822429906542056, "recall": 0.6700507614213198, "f1": 0.7384615384615385, "macro_f1": 0.6552719591621194, "accuracy": 0.6753472222222222, "balanced_accuracy": 0.6784319741172533, "specificity": 0.6868131868131868, "roc_auc": 0.7420929324482624, "average_precision": 0.8766007847344726, "tn": 125, "fp": 57, "fn": 130, "tp": 264, "threshold": 0.5, "images": 576, "model": "mobilenet_v2", "teacher": "vgg16", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.12012923536600809, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.9599761974807848e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "cmmd__feature__nasnet_mobile__from__densenet121", "dataset": "cmmd", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-19", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-19.data-00000-of-00001": "e55061051e9f9dabc3614d18521666bd3efd1e8e2a1a67ec5f466302ed6febc3", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-19.index": "bd6d6472c16f6898ac55a6a8dcf21e2d4310ae6e9ff2b6d98100c0210aa74a78"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42/config.json": "015fab1a4226c90d7ec799263ec9e30daa9d20fbaefca2edf1c3381d30d7e808", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "c69d02685b3c342da6c7ab699f12f1b8ffd51befe6918c8cb3ce8ceff5dcab1b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/nasnet_mobile__from__densenet121/test_metrics.json": "575d999bca337323b859764ce06ad71885dc9978cec931064970567d83b6d0ec", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.7802197802197802, "recall": 0.7208121827411168, "f1": 0.7493403693931399, "macro_f1": 0.6335534334275343, "accuracy": 0.6701388888888888, "balanced_accuracy": 0.6406258715903386, "specificity": 0.5604395604395604, "roc_auc": 0.7252607798293078, "average_precision": 0.8586601973700209, "tn": 102, "fp": 80, "fn": 110, "tp": 284, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "teacher": "densenet121", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.1347498974248619, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.96905517949142e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "cmmd__feature__nasnet_mobile__from__none", "dataset": "cmmd", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-10", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-10.data-00000-of-00001": "d76dbb69d2e6c8889bacfbb97a5148bc64643fd5699bf42be30acdbe246f2490", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-10.index": "33c4352658d6971936a49bbc5e57dbf3dbc6e5341fca4fe53cb4e78f315926a7"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42/config.json": "84fb9053e6df544e7682c3740e231d9a0d2d9f60d97f14be96ec8e90a3abf870", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42/COMPLETE.json": "f54d0ef9209d01def6d538a489161b785f86743f56c414b864344aaefad6a2e2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/nasnet_mobile__from__none/test_metrics.json": "74478b369b86d4e289ec442b26fe725a997372d57973026f10aee2289c54b45c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.7493975903614458, "recall": 0.7893401015228426, "f1": 0.7688504326328801, "macro_f1": 0.6118304641298511, "accuracy": 0.6753472222222222, "balanced_accuracy": 0.6089557650471356, "specificity": 0.42857142857142855, "roc_auc": 0.6965610531600379, "average_precision": 0.831100424567771, "tn": 78, "fp": 104, "fn": 83, "tp": 311, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "teacher": "none", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 2.969818113651712e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "cmmd__feature__nasnet_mobile__from__vgg16", "dataset": "cmmd", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13.data-00000-of-00001": "c80394b9c4c18fe7e1d17871c912d1b6150940e503c745160fd0bdc0f8682532", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-13.index": "ead716ffdfb681f39e132f2287b0827ef62634dbbae60f11e2559c1da9c6be7e"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/protocol.json": "994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42/config.json": "ceb47867c479168c8193c21c6914b7a04dc5f1c7beeabacd15203bb4d4ba4bad", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "bf22344b0319f5eb0525babfee8fc824a16b8a5c474e075462d1c31e34c7070c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/nasnet_mobile__from__vgg16/test_metrics.json": "95b8b55f3417784af290f9dce02d63e1c23a603c11afc8589665673e71561629", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/cmmd/evaluation_lock.json": "d8adb145d6795d6ebd0f1dfb4ee225a9da1a4666e143770d6196ba0c5c3ece41"}, "metrics": {"precision": 0.7714285714285715, "recall": 0.7538071065989848, "f1": 0.7625160462130937, "macro_f1": 0.6332687469671366, "accuracy": 0.6788194444444444, "balanced_accuracy": 0.6351453115412506, "specificity": 0.5164835164835165, "roc_auc": 0.7128074970714564, "average_precision": 0.850910881290023, "tn": 94, "fp": 88, "fn": 97, "tp": 297, "threshold": 0.5, "images": 576, "model": "nasnet_mobile", "teacher": "vgg16", "dataset": "cmmd", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.1977017457711479, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/cmmd/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.980194091506405e-08, "class_disagreements": 0}, "report": "B06C_cmmd_seed42_test_report.zip", "report_sha256": "b5d73743e218c7f335d5ea37e680020181fc5bfff48a469a1ed2be41c6110ab5"}, {"run_id": "inbreast__feature__mobilenet_v2__from__densenet121", "dataset": "inbreast", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-20", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-20.data-00000-of-00001": "2ab4298228fe8de567804299f5ddbc8cd99b4d00c2f0bc74da1d7c2617d89287", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-20.index": "3812213553ee4062d8261195bc86afe30a3467a8b86c4dfb44f206daa1db0dfd"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42/config.json": "eb1da87bbcf14145a9b9714a39e0cfa1c5054da06a438ebfe431add8ea35142a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "0fe44cd5840c1e62ae8c3283f002814dc6ea90c9613042ca54d9b42adadd783f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/mobilenet_v2__from__densenet121/test_metrics.json": "222fc13407b0ea7fe94ece3c08d5541a49ff1f2bc46243cd4757a9d060e34d62", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.3333333333333333, "recall": 0.5714285714285714, "f1": 0.42105263157894735, "macro_f1": 0.6500867553499132, "accuracy": 0.8, "balanced_accuracy": 0.7023809523809523, "specificity": 0.8333333333333334, "roc_auc": 0.8422619047619048, "average_precision": 0.39736961451247166, "tn": 40, "fp": 8, "fn": 3, "tp": 4, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "teacher": "densenet121", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.37003424332926393, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 1.938400273626684e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "inbreast__feature__mobilenet_v2__from__none", "dataset": "inbreast", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-24", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-24.data-00000-of-00001": "469c7497ad6d7b6d34b853e12a4997f1b4eec720f6cac335dee43558061d3aa9", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-24.index": "2c5f10dc41173a7e4e3f19e2e1b561d6eda4bc51ab7ccd60c0d8b28a2674927d"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42/config.json": "480b25a6406d8037fe5a0e74785c70d56d1cce89d7109ddd19d79a97ffa8743c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42/COMPLETE.json": "858f64b58128a4f54f461b55548f2cc13068a504d02a3ebcf3657fd659179a9e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/mobilenet_v2__from__none/test_metrics.json": "fd0bccada7dc910d4ceba4b666a03cca9000ffea450f6f5bdd6a5d161d9cc6ff", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.35714285714285715, "recall": 0.7142857142857143, "f1": 0.47619047619047616, "macro_f1": 0.6762974852862493, "accuracy": 0.8, "balanced_accuracy": 0.7633928571428572, "specificity": 0.8125, "roc_auc": 0.8601190476190476, "average_precision": 0.4232615268329555, "tn": 39, "fp": 9, "fn": 2, "tp": 5, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "teacher": "none", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 2.7794647250445337e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "inbreast__feature__mobilenet_v2__from__vgg16", "dataset": "inbreast", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-20", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-20.data-00000-of-00001": "1062653a66ec3ed99ae5f2f6afc8a7b24d98858dd87673437f94aeb536c9f629", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-20.index": "94e5821620d9de1362121b708992f1f085b834e2e999087936d4e53569275e4c"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42/config.json": "7fd5f8e90cd65f0e5ab5f01fe909729a65cc8190da905e0fab4da4b879eec7c0", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "75c248e2a342949fe45b0fa446db3d4ed317397f62d4ed4301d9a49ce108a3cd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/mobilenet_v2__from__vgg16/test_metrics.json": "f94e7f28752ac19f4b2fffee919029e3d9c75fb7d648a9e2b7328bb8e279b6a2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.38461538461538464, "recall": 0.7142857142857143, "f1": 0.5, "macro_f1": 0.6944444444444444, "accuracy": 0.8181818181818182, "balanced_accuracy": 0.7738095238095238, "specificity": 0.8333333333333334, "roc_auc": 0.8392857142857142, "average_precision": 0.45960111317254176, "tn": 40, "fp": 8, "fn": 2, "tp": 5, "threshold": 0.5, "images": 55, "model": "mobilenet_v2", "teacher": "vgg16", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.07550292368483152, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.6530456498363723e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "inbreast__feature__nasnet_mobile__from__densenet121", "dataset": "inbreast", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23.data-00000-of-00001": "d069f0acd71581d0b71e55d55aec49b86977777afd404b012d4e57f57a7d0f59", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-23.index": "a1aeb438381492626531f52881ccc6b2e1c091478bbdf60c2d78dce6c1d58372"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42/config.json": "1a2f5ca656439b3231965b73dc7ee1448da81bbc627bb47af44612bfc2a9ca2e", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "fe4a51a734853ca7aa964ef0c5a053c4998beceb8581f2988027414cc70c539a", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/nasnet_mobile__from__densenet121/test_metrics.json": "28436c2f3b65e98e5e8b1ac3d9c41009c4c4d8d73dc790daee7a66820ba4595f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.5, "recall": 0.2857142857142857, "f1": 0.36363636363636365, "macro_f1": 0.6464646464646464, "accuracy": 0.8727272727272727, "balanced_accuracy": 0.6220238095238095, "specificity": 0.9583333333333334, "roc_auc": 0.5803571428571428, "average_precision": 0.3485101421761829, "tn": 46, "fp": 2, "fn": 5, "tp": 2, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "teacher": "densenet121", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.6395188386128099, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 1.5612030002110092e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "inbreast__feature__nasnet_mobile__from__none", "dataset": "inbreast", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-12", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-12.data-00000-of-00001": "4b6cd30d304157e0b17c886de19d3b9444d1dcef2a10c7339e121a79f0b0c232", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-12.index": "31f7a5db54a6596fa327cb04e21ab0ddf08a6b6dad022f0e5517c0ba11bd95b3"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42/config.json": "30d1a8d844f5987fc1bf12edc0a12891928608e45627b8092eea7dce3897fc07", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42/COMPLETE.json": "e04056fbc448366c9daf5081742bf4e8f725c4a59f78dff4c936fac45267e98c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/nasnet_mobile__from__none/test_metrics.json": "8226daf606446bad8ece496b883c7c2cd782f5c97bb8703d78b727e6c6931f09", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.23076923076923078, "recall": 0.42857142857142855, "f1": 0.3, "macro_f1": 0.5722222222222222, "accuracy": 0.7454545454545455, "balanced_accuracy": 0.6101190476190476, "specificity": 0.7916666666666666, "roc_auc": 0.5803571428571428, "average_precision": 0.3698823913533746, "tn": 38, "fp": 10, "fn": 4, "tp": 3, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "teacher": "none", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 2.8137207008782642e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "inbreast__feature__nasnet_mobile__from__vgg16", "dataset": "inbreast", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-16", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-16.data-00000-of-00001": "54f425fd0d6c120b66b2a59e50248a6629a4b84acbbb818034efa824a708f36c", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-16.index": "725eca38f42c61825cca8cad78b87b88f2e4377bb5a15e082b8a2a204355f675"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/protocol.json": "fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42/config.json": "fa4d39634389a195a977108b4266bc00c1a5850deb7b3e470c383f3ddf71c121", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "f32ac38b692da60dbfebde3763da432205aee39f38e08b3033b0e8de411e36aa", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/nasnet_mobile__from__vgg16/test_metrics.json": "d6eb0d2b1fa6b756cae08f064ab8eef850d3979e71b6c1926a2724ae4734ddba", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/inbreast/evaluation_lock.json": "1023b7a402a6dd7ff549c8a9c5e1bc7fce8a5337a8efc4988ac9f0caed79857a"}, "metrics": {"precision": 0.42857142857142855, "recall": 0.42857142857142855, "f1": 0.42857142857142855, "macro_f1": 0.6726190476190476, "accuracy": 0.8545454545454545, "balanced_accuracy": 0.6726190476190476, "specificity": 0.9166666666666666, "roc_auc": 0.6309523809523808, "average_precision": 0.33983023948288615, "tn": 44, "fp": 4, "fn": 4, "tp": 3, "threshold": 0.5, "images": 55, "model": "nasnet_mobile", "teacher": "vgg16", "dataset": "inbreast", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.0733818420089392, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/inbreast/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.8707885735279604e-08, "class_disagreements": 0}, "report": "B06C_inbreast_seed42_test_report.zip", "report_sha256": "80967abf27d1b173ffdb77bd4de87c27b315a47e90c6f4452eafe288e866fd69"}, {"run_id": "mias__feature__mobilenet_v2__from__densenet121", "dataset": "mias", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.data-00000-of-00001": "2b1058c955c7c63d884c1e0d8193b867476355421c910d4df89cafc3e0ee9ba2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42/finetune/best/ckpt-19.index": "67d9e76017631d2e753bc0197209113b842f6f998888c748a577b4fba355a092"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42/config.json": "3dba5e2d3d551ae65d1895cb79103a6c3f8447fe2c280a716b2844f1b08cc9bd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42/COMPLETE.json": "94de9b54dcb5fd7996419e131c9b339d96a56cc93772caaac6bbcf3f1b4b4856", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/mobilenet_v2__from__densenet121/test_metrics.json": "130c8e655699eebfd55d8fadf34eb4af85472096d7e2881e87d01ec3e6224a6b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.45454545454545453, "recall": 0.625, "f1": 0.5263157894736842, "macro_f1": 0.4631578947368421, "accuracy": 0.47058823529411764, "balanced_accuracy": 0.47916666666666663, "specificity": 0.3333333333333333, "roc_auc": 0.40277777777777785, "average_precision": 0.44426499766573296, "tn": 3, "fp": 6, "fn": 3, "tp": 5, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "teacher": "densenet121", "dataset": "mias", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 2.0927532223678136, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.9343414253979464e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}, {"run_id": "mias__feature__mobilenet_v2__from__none", "dataset": "mias", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-1", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-1.data-00000-of-00001": "589e1864abd2a6a4555f45008dd287f401ac9f7bb46597c09999287a9a1c1748", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-1.index": "7eecfe4364b73b4dcdb17cf8c5cbcee54b1e5699a1bc779f67f984c877369db5"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42/config.json": "0ac93d71ce607da5b43a11a05a2553c758fb19bb1e63ad5f097f6c4a5fe11fc4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42/COMPLETE.json": "e6b1988b8d48ff44709227cde1c482b47660911c5d8bffa590da51aafdc932cc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/mobilenet_v2__from__none/test_metrics.json": "643b7ea36483d06ea6b77ae0bec8d9dc386103a20a753a85fadf560fc809778b", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.5, "recall": 0.625, "f1": 0.5555555555555556, "macro_f1": 0.5277777777777778, "accuracy": 0.5294117647058824, "balanced_accuracy": 0.5347222222222222, "specificity": 0.4444444444444444, "roc_auc": 0.375, "average_precision": 0.42219158496732023, "tn": 4, "fp": 5, "fn": 3, "tp": 5, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "teacher": "none", "dataset": "mias", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__none/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "none", "passed": true, "max_abs_difference": 2.2823333778454472e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}, {"run_id": "mias__feature__mobilenet_v2__from__vgg16", "dataset": "mias", "pipeline": "feature", "student": "mobilenet_v2", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-1", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-1.data-00000-of-00001": "39641c6d0ece8b5f4f51e1c10d8eacf6837b177ffc4cdb83285482c753959863", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42/finetune/best/ckpt-1.index": "dc82cad01e474f8dcf6c25a0e52336133e53d842e9988f12d30d20fd69efdc94"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42/config.json": "32dca61b64b657870ea470392423a92cdebd0c3535c6d67e3d47faa827dafbab", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42/COMPLETE.json": "df6c10579454987bbf1bc61ee5348647d01faf762297f9719b0bb7cfd3925996", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/mobilenet_v2__from__vgg16/test_metrics.json": "88f77c6a50d21554aafbd037503e3dff1c1380139a02bfbee0f974c65f344459", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.5, "recall": 0.625, "f1": 0.5555555555555556, "macro_f1": 0.5277777777777778, "accuracy": 0.5294117647058824, "balanced_accuracy": 0.5347222222222222, "specificity": 0.4444444444444444, "roc_auc": 0.375, "average_precision": 0.42219158496732023, "tn": 4, "fp": 5, "fn": 3, "tp": 5, "threshold": 0.5, "images": 17, "model": "mobilenet_v2", "teacher": "vgg16", "dataset": "mias", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.07946441011281352, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/mobilenet_v2__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "mobilenet_v2", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.219161987504492e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}, {"run_id": "mias__feature__nasnet_mobile__from__densenet121", "dataset": "mias", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "densenet121", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-2", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-2.data-00000-of-00001": "3ef9b4047468be2df16f925a7b1019cfb5837fbd3be6575dc65865728f24bd90", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42/finetune/best/ckpt-2.index": "be67484240356afc45d7edb197d2390b297c334e78db7cc12474b855df6f5721"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42/config.json": "b375d83a08971b0050c45904d8da82bb42e26d07574675dc3dbdc497b5fe49cd", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42/COMPLETE.json": "2335864f0f82ab5da5bbe15a2136fa66174c1e15c155fa01baa212dfbfc0a9ab", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/nasnet_mobile__from__densenet121/test_metrics.json": "294c711c3bd804b04a3847adc3fd05e302cb0876a3be0c0e9747d56e76fd27a4", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.2, "recall": 0.125, "f1": 0.15384615384615385, "macro_f1": 0.315018315018315, "accuracy": 0.35294117647058826, "balanced_accuracy": 0.3402777777777778, "specificity": 0.5555555555555556, "roc_auc": 0.2916666666666667, "average_precision": 0.3914346682729036, "tn": 5, "fp": 4, "fn": 7, "tp": 1, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "teacher": "densenet121", "dataset": "mias", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.9835628031340428, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__densenet121/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "densenet121", "passed": true, "max_abs_difference": 2.192764281261361e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}, {"run_id": "mias__feature__nasnet_mobile__from__none", "dataset": "mias", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "none", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-1", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-1.data-00000-of-00001": "b2c7729739d7acd2d0c31c66078e78185936bced3aa1611e4924c32d62421a77", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-1.index": "8972be9ece2a2ad490b1291e2b00635c70633119414ea4e07dea17609f059b7e"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42/config.json": "3cff9a8084913b5abf8fae84b19f11b8009d2c124deda0a5dc2e0e5ffeecb8e9", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42/COMPLETE.json": "48802486ae91b6336cb8e835c871259deaa7d3819f983cd7a62d68f364298bb1", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/nasnet_mobile__from__none/test_metrics.json": "d418f230f26951959de21ad34fdb27bca4e2ce9b60d63054d37dfa8c317bf22f", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.25, "recall": 0.125, "f1": 0.16666666666666666, "macro_f1": 0.356060606060606, "accuracy": 0.4117647058823529, "balanced_accuracy": 0.3958333333333333, "specificity": 0.6666666666666666, "roc_auc": 0.2777777777777778, "average_precision": 0.3833245492252845, "tn": 6, "fp": 3, "fn": 7, "tp": 1, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "teacher": "none", "dataset": "mias", "training_seed": 42, "split": "test", "method": "supervised_control", "lambda_feature": 0.0, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__none/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "none", "passed": true, "max_abs_difference": 2.572937007894183e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}, {"run_id": "mias__feature__nasnet_mobile__from__vgg16", "dataset": "mias", "pipeline": "feature", "student": "nasnet_mobile", "teacher": "vgg16", "training_seed": 42, "checkpoint": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-15", "checkpoint_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-15.data-00000-of-00001": "c34cad3ab7538a945172327be4bddd4e80cde581062463636a63661984483bd5", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42/finetune/best/ckpt-15.index": "97f800882d15663935846d43b13c8e334855515e68b2a29cec813fd1e64e6978"}, "metadata_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/protocol.json": "e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42/config.json": "b58688b7826ffe20ed077a8e16f726bf91b7fd7c4e4a5991c108de6692935c76", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42/COMPLETE.json": "555a10c544641d98f9fb9df011dfd6e19f4d868fc765ae9f8bc7a094173e4e14", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/inference_helpers.py": "afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876"}, "performance_sha256": {"/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/nasnet_mobile__from__vgg16/test_metrics.json": "adea9305518593e7b64296a8396c8505dfa0f6dc05abc69dfc1e1ac58ae29fe2", "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06c_feature_kd_test_v1/mias/evaluation_lock.json": "8cf67de07a8117cca86aa7eeebb015276ff2838f94c414931dcf1b10c16fbc58"}, "metrics": {"precision": 0.2857142857142857, "recall": 0.25, "f1": 0.26666666666666666, "macro_f1": 0.343859649122807, "accuracy": 0.35294117647058826, "balanced_accuracy": 0.3472222222222222, "specificity": 0.4444444444444444, "roc_auc": 0.36111111111111116, "average_precision": 0.43543851981351983, "tn": 4, "fp": 5, "fn": 6, "tp": 2, "threshold": 0.5, "images": 17, "model": "nasnet_mobile", "teacher": "vgg16", "dataset": "mias", "training_seed": 42, "split": "test", "method": "ssid_inspired_feature_kd", "lambda_feature": 0.04202358839226118, "source_run": "/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1/mias/nasnet_mobile__from__vgg16/seed_42"}, "previous_validation_audit": {"student": "nasnet_mobile", "teacher": "vgg16", "passed": true, "max_abs_difference": 2.422790523759488e-08, "class_disagreements": 0}, "report": "B06C_mias_seed42_test_report.zip", "report_sha256": "3db0279a28b2ac79f31ee9932642508286e84a76169200d089eec059b97457fb"}]')
BUILDER_SOURCE="def build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-05, stage='finetune'):\n    keras.utils.set_random_seed(seed)\n    _, constructor, _ = REGISTRY[key]\n    base = getattr(keras.applications, constructor)(include_top=False, weights=weights, input_shape=(size, size, 3), pooling='avg')\n    base.trainable = stage == 'finetune'\n    for layer in base.layers:\n        if isinstance(layer, keras.layers.BatchNormalization):\n            layer.trainable = False\n            layer.fused = False\n    inp = keras.Input((size, size, 3), name='preprocessed_image')\n    x = base(inp, training=False)\n    x = keras.layers.Dropout(0.2, seed=seed, name='head_dropout')(x)\n    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)\n    model = keras.Model(inp, output, name=key)\n    return model\n"
WORKER_SOURCE='"""B08D: forward-only checkpoint audit/export and isolated GPU profiling."""\nimport os,sys,json,time,hashlib,gc,platform,subprocess\nfrom pathlib import Path\nMODE=sys.argv[1]\nif MODE==\'audit\':os.environ[\'CUDA_VISIBLE_DEVICES\']=\'-1\'\nos.environ[\'TF_DETERMINISTIC_OPS\']=\'1\'\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\']=\'2\'\nimport numpy as np\nimport tensorflow as tf\nimport tf_keras as keras\nREGISTRY={\'mobilenet_v2\':[\'student\',\'MobileNetV2\',\'mobilenet_v2\'],\'nasnet_mobile\':[\'student\',\'NASNetMobile\',\'nasnet\']}\nEXPECTED={\'mobilenet_v2\':2259265,\'nasnet_mobile\':4270773}\ndef sha(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef save(p,o):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+\'.tmp\')\n t.write_text(json.dumps(o,indent=2,allow_nan=False));os.replace(t,p)\ndef digest_weights(m):\n h=hashlib.sha256()\n for v in m.weights:\n  a=v.numpy();h.update(str(a.shape).encode());h.update(str(a.dtype).encode());h.update(a.tobytes())\n return h.hexdigest()\ndef make_model(c):\n ns={\'tf\':tf,\'keras\':keras,\'REGISTRY\':REGISTRY}\n if sha(c[\'builder_path\'])!=c[\'builder_sha256\']:raise RuntimeError(\'Builder changed\')\n exec(compile(Path(c[\'builder_path\']).read_text(),c[\'builder_path\'],\'exec\'),ns)\n m=ns[\'build_model\'](c[\'student\'],size=512,seed=42,weights=None,stage=\'head\')\n if m.count_params()!=EXPECTED[c[\'student\']]:raise RuntimeError(\'Parameter count mismatch\')\n if m.output_shape!=(None,1) or m.get_layer(\'probability\').activation.__name__!=\'sigmoid\':raise RuntimeError(\'Unexpected head\')\n if any(\'projector\' in l.name.lower() or \'teacher\' in l.name.lower() for l in m.submodules):raise RuntimeError(\'Training component in deployment\')\n m.trainable=False\n return m\ndef synthetic(c,batch):\n raw=np.random.default_rng(42).uniform(0,255,(batch,512,512,1)).astype(\'float32\')\n raw=np.repeat(raw,3,axis=-1)\n return getattr(keras.applications,REGISTRY[c[\'student\']][2]).preprocess_input(tf.convert_to_tensor(raw))\ndef environment():\n g=tf.config.list_physical_devices(\'GPU\')\n return dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,\n  gpu=tf.config.experimental.get_device_details(g[0]) if g else {},build=tf.sysconfig.get_build_info(),\n  tf32_enabled=tf.config.experimental.tensor_float_32_execution_enabled(),precision=\'float32\',\n  nvidia_smi=subprocess.check_output([\'nvidia-smi\'],text=True))\ndef weighted_macs(layer):\n    """Per-image Conv/Dense multiply-accumulates, excluding all other operations."""\n    sh=layer.output_shape\n    if isinstance(sh,list):raise ValueError(\'Multiple outputs not supported: \'+layer.name)\n    dims=sh[1:]\n    if any(x is None for x in dims):raise ValueError(\'Unknown static shape: \'+layer.name)\n    outputs=int(np.prod(dims))\n    L=keras.layers\n    if isinstance(layer,L.SeparableConv2D):\n        kh,kw,inc,mult=map(int,layer.depthwise_kernel.shape)\n        spatial=int(np.prod(dims[:-1]))\n        return spatial*inc*mult*kh*kw + outputs*inc*mult\n    if isinstance(layer,L.DepthwiseConv2D):\n        kh,kw,_,_=map(int,layer.depthwise_kernel.shape)\n        return outputs*kh*kw\n    if isinstance(layer,L.Conv2D):\n        kh,kw,inc_per_group,_=map(int,layer.kernel.shape)\n        return outputs*kh*kw*inc_per_group\n    if isinstance(layer,L.Dense):return outputs*int(layer.kernel.shape[0])\n    raise TypeError(layer.__class__.__name__)\n\ndef count_macs(model):\n    seen=set();rows=[];excluded={}\n    supported=(keras.layers.Conv2D,keras.layers.DepthwiseConv2D,keras.layers.SeparableConv2D,keras.layers.Dense)\n    for layer in model.submodules:\n        if id(layer) in seen or isinstance(layer,keras.Model):continue\n        seen.add(id(layer))\n        if isinstance(layer,supported):\n            if len(layer._inbound_nodes)!=1:\n                raise RuntimeError(\'Shared/called-multiple-times layer needs operation-level count: \'+layer.name)\n            rows.append(dict(layer=layer.name,type=type(layer).__name__,macs=weighted_macs(layer)))\n        else:\n            k=type(layer).__name__;excluded[k]=excluded.get(k,0)+1\n            if layer.weights and not isinstance(layer,keras.layers.BatchNormalization):\n                raise RuntimeError(\'Unaccounted weighted layer: \'+k)\n    if not rows:raise RuntimeError(\'MAC count empty\')\n    return sum(r[\'macs\'] for r in rows),rows,excluded\n\ndef main(mode,config_path):\n c=json.loads(Path(config_path).read_text());out=Path(c[\'job_dir\']);out.mkdir(parents=True,exist_ok=True)\n keras.utils.set_random_seed(42);keras.mixed_precision.set_global_policy(\'float32\')\n tf.config.experimental.enable_tensor_float_32_execution(False)\n tf.config.experimental.enable_op_determinism();tf.config.optimizer.set_jit(False)\n if mode==\'audit\':\n  if tf.config.list_physical_devices(\'GPU\'):raise RuntimeError(\'Export audit must run on CPU\')\n  for p,h in c[\'local_checkpoint_hashes\'].items():\n   if sha(p)!=h:raise RuntimeError(\'Local checkpoint hash mismatch: \'+p)\n  m=make_model(c);base=next(l for l in m.layers if isinstance(l,keras.Model))\n  ck=tf.train.Checkpoint(backbone=base,classifier=m.get_layer(\'probability\'))\n  status=ck.restore(c[\'local_checkpoint\']);status.assert_existing_objects_matched();status.expect_partial()\n  # Extra projector/optimizer entries in the source checkpoint are intentionally not restored.\n  if not all(np.isfinite(v.numpy()).all() for v in m.weights):raise RuntimeError(\'Non-finite checkpoint weights\')\n  before=digest_weights(m);x=synthetic(c,2);p=m(x,training=False).numpy()\n  if not np.isfinite(p).all():raise RuntimeError(\'Non-finite audit prediction\')\n  m.save_weights(c[\'export_path\']);weight_bytes=sum(int(np.prod(v.shape))*tf.as_dtype(v.dtype).size for v in m.weights)\n  del status,ck,base,m;keras.backend.clear_session();gc.collect()\n  m=make_model(c);m.load_weights(c[\'export_path\']);q=m(x,training=False).numpy()\n  if digest_weights(m)!=before:raise RuntimeError(\'Export changed model weights\')\n  np.testing.assert_allclose(p,q,rtol=1e-6,atol=1e-7)\n  np.testing.assert_array_equal(p>=.5,q>=.5)\n  record=dict(status=\'complete\',run_id=c[\'run_id\'],configuration_sha256=sha(config_path),\n   parameters=m.count_params(),raw_weight_MiB=weight_bytes/2**20,h5_weight_MiB=Path(c[\'export_path\']).stat().st_size/2**20,\n   export_sha256=sha(c[\'export_path\']),deployment_weight_sha256=before,\n   max_export_probability_difference=float(np.max(abs(p-q))),export_class_disagreements=int(np.sum((p>=.5)!=(q>=.5))),\n   previous_validation_restore_passed=True,validation_rerun=False,synthetic_export_roundtrip_passed=True,\n   note=\'Checkpoint bytes locked to reviewed B05C/B06C validation/test audit; no new image inference.\')\n  save(out/\'audit_result.json\',record);print(\'PASS audit/export\',c[\'run_id\'],flush=True);return\n gpus=tf.config.list_physical_devices(\'GPU\')\n if not gpus:raise RuntimeError(\'GPU required\')\n tf.config.set_visible_devices(gpus[0],\'GPU\');tf.config.experimental.set_memory_growth(gpus[0],True)\n audit=json.loads(Path(c[\'audit_path\']).read_text())\n if sha(c[\'export_path\'])!=audit[\'export_sha256\']:raise RuntimeError(\'Deployment export changed\')\n with tf.device(\'/GPU:0\'):\n  m=make_model(c);m.load_weights(c[\'export_path\']);x=synthetic(c,c[\'batch\'])\n if digest_weights(m)!=audit[\'deployment_weight_sha256\']:raise RuntimeError(\'GPU model weights differ from audited export\')\n if \'GPU\' not in x.device or any(\'GPU\' not in v.device for v in m.weights):raise RuntimeError(\'Model/input not entirely on GPU\')\n mac,macrows,excluded=count_macs(m);save(out/\'macs_by_layer.json\',macrows)\n with (out/\'model_summary.txt\').open(\'w\') as f:m.summary(print_fn=lambda line:f.write(line+\'\\n\'),expand_nested=True)\n @tf.function(input_signature=[tf.TensorSpec([c[\'batch\'],512,512,3],tf.float32)],autograph=False,jit_compile=False)\n def forward(v):return m(v,training=False)\n for _ in range(c[\'warmup\']):p=forward(x).numpy()\n if p.shape!=(c[\'batch\'],1) or not np.isfinite(p).all() or not ((p>=0)&(p<=1)).all():raise RuntimeError(\'Invalid forward probabilities\')\n del p;gc.collect();baseline=tf.config.experimental.get_memory_info(\'GPU:0\')[\'current\']\n tf.config.experimental.reset_memory_stats(\'GPU:0\');timings=[];trials=[]\n for trial in range(c[\'trials\']):\n  seconds=[]\n  for iteration in range(c[\'iterations\']):\n   start=time.perf_counter_ns();p=forward(x).numpy();elapsed=(time.perf_counter_ns()-start)/1e9\n   seconds.append(elapsed);timings.append(dict(trial=trial+1,iteration=iteration+1,batch_seconds=elapsed))\n  mean=float(np.mean(seconds));trials.append(dict(trial=trial+1,batch_mean_ms=mean*1000,throughput_images_s=c[\'batch\']/mean))\n  print(c[\'run_id\'],\'batch\',c[\'batch\'],\'trial\',trial+1,\'ms\',round(mean*1000,3),flush=True)\n mem=tf.config.experimental.get_memory_info(\'GPU:0\');env=environment();vals=np.array([r[\'batch_seconds\']*1000 for r in timings]);means=np.array([r[\'batch_mean_ms\'] for r in trials])\n save(out/\'environment.json\',env);save(out/\'timings.json\',timings);save(out/\'trials.json\',trials)\n result={k:c[k] for k in [\'run_id\',\'dataset\',\'pipeline\',\'student\',\'teacher\',\'training_seed\',\'batch\',\'session_id\']}\n result.update(status=\'complete\',configuration_sha256=sha(config_path),parameters=m.count_params(),\n  parameter_millions=m.count_params()/1e6,conv_dense_gmacs=mac/1e9,conv_dense_macs=mac,\n  raw_weight_MiB=audit[\'raw_weight_MiB\'],h5_weight_MiB=audit[\'h5_weight_MiB\'],\n  batch_latency_mean_ms=float(vals.mean()),trial_mean_latency_sd_ms=float(means.std(ddof=1)),\n  batch_latency_p95_ms=float(np.quantile(vals,.95)),throughput_images_s=float(c[\'batch\']*1000/vals.mean()),\n  amortized_ms_per_image=float(vals.mean()/c[\'batch\']),tf_active_baseline_MiB=baseline/2**20,\n  tf_active_peak_MiB=mem[\'peak\']/2**20,tf_extra_peak_over_baseline_MiB=max(0,mem[\'peak\']-baseline)/2**20,\n  gpu=env[\'gpu\'].get(\'device_name\'),tensorflow=tf.__version__,tf_keras=keras.__version__,\n  deployment_weight_sha256=audit[\'deployment_weight_sha256\'],excluded_mac_operations=excluded)\n save(out/\'result.json\',result)\nif __name__==\'__main__\':\n try:main(MODE,sys.argv[2])\n except Exception:\n  import traceback\n  traceback.print_exc();sys.exit(1)\n'
ORCHESTRATION_SHA256='02a5195a85bfeff417ff1e560bac86311deb0d906e5ae84b3e9bced8d97bd374'
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
def objsha(o):return hashlib.sha256(json.dumps(o,sort_keys=True,allow_nan=False).encode()).hexdigest()
def read(p):return json.loads(Path(p).read_text())
def save(p,o):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+'.tmp')
    t.write_text(json.dumps(o,indent=2,allow_nan=False));os.replace(t,p)
def guard(p,o):
    if Path(p).exists() and read(p)!=o:raise RuntimeError('Existing file differs: '+str(p))
    save(p,o)
def relocate(s):
    prefix='/content/drive/MyDrive/Research'
    if not s.startswith(prefix+'/'):raise ValueError('Unexpected source root: '+s)
    return RESEARCH/s[len(prefix)+1:]
def verify_files(expected):
    for original,digest in expected.items():
        p=relocate(original)
        if not p.is_file():raise FileNotFoundError(str(p))
        if sha(p)!=digest:raise RuntimeError('Source differs from reviewed report: '+str(p))
def checkpoint_copy(run):
    dest=LOCAL/'checkpoints'/run['run_id'];dest.mkdir(parents=True,exist_ok=True);hashes={}
    for original,digest in run['checkpoint_sha256'].items():
        src=relocate(original);target=dest/src.name
        if not target.is_file() or sha(target)!=digest:shutil.copyfile(src,target)
        if sha(target)!=digest:raise RuntimeError('Checkpoint copy failed: '+str(target))
        hashes[str(target)]=digest
    return str(dest/Path(run['checkpoint']).name),hashes

def run_process(mode,config_path,log_path):
    env=os.environ.copy();env['PYTHONUNBUFFERED']='1';env['TF_DETERMINISTIC_OPS']='1'
    if mode=='audit':env['CUDA_VISIBLE_DEVICES']='-1'
    else:env['CUDA_VISIBLE_DEVICES']='0'
    cmd=[sys.executable,str(WORKER_PATH),mode,str(config_path)]
    started=time.monotonic()
    with Path(log_path).open('w') as log:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        try:
            while process.poll() is None:
                if time.monotonic()-started>JOB_TIMEOUT_SECONDS:raise TimeoutError('Worker timeout; see '+str(log_path))
                ready,_,_=select.select([process.stdout],[],[],15)
                if ready:
                    line=process.stdout.readline();log.write(line);log.flush()
                    if 'trial ' in line or 'PASS ' in line:print(line.rstrip(),flush=True)
                else:print('  running',mode,'elapsed',int(time.monotonic()-started),'s',flush=True)
            rest=process.stdout.read();log.write(rest);log.flush()
            if process.returncode:
                tail=Path(log_path).read_text(errors='replace')[-4500:]
                raise RuntimeError(f'Worker exit {process.returncode}\n{tail}')
        finally:
            if process.poll() is None:
                process.terminate()
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:process.kill();process.wait()

def config_matches(a,b):
    # A new Colab session can continue completed jobs on the same declared environment.
    return {k:v for k,v in a.items() if k!='session_id'}=={k:v for k,v in b.items() if k!='session_id'}
def finished_job(folder,cfg):
    folder=Path(folder);rp=folder/'result.json';cp=folder/'config.json';done=folder/'COMPLETE.json'
    if not done.is_file():return False
    if not cp.is_file() or not config_matches(read(cp),cfg):raise RuntimeError('Job configuration changed: '+str(folder))
    d=read(done)
    for name,digest in d['outputs'].items():
        if not (folder/name).is_file() or sha(folder/name)!=digest:raise RuntimeError('Completed job changed: '+str(folder/name))
    r=read(rp)
    if r['configuration_sha256']!=sha(cp) or r['status']!='complete':raise RuntimeError('Invalid completion: '+str(folder))
    return True

def summary_tables(results,performance):
    keys=['run_id','dataset','pipeline','student','teacher','training_seed']
    if not results:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    df=pd.DataFrame(results)
    if df.duplicated(['run_id','batch']).any():raise RuntimeError('Duplicate jobs')
    b1=df[df.batch==1].copy();b8=df[df.batch==8].copy()
    a=b1[keys+['parameters','conv_dense_gmacs','raw_weight_MiB','h5_weight_MiB','batch_latency_mean_ms','trial_mean_latency_sd_ms','tf_active_peak_MiB','deployment_weight_sha256','gpu','tensorflow','tf_keras','session_id']].rename(columns={'batch_latency_mean_ms':'latency_b1_ms','trial_mean_latency_sd_ms':'latency_b1_trial_sd_ms','tf_active_peak_MiB':'peak_b1_MiB','session_id':'session_b1'})
    b=b8[['run_id','throughput_images_s','tf_active_peak_MiB','parameters','conv_dense_gmacs','deployment_weight_sha256','session_id']].rename(columns={'throughput_images_s':'throughput_b8_images_s','tf_active_peak_MiB':'peak_b8_MiB','session_id':'session_b8'})
    common=b1[['run_id','parameters','conv_dense_gmacs','deployment_weight_sha256']].merge(b,on='run_id',suffixes=('_b1','_b8'),validate='one_to_one')
    for col in ['parameters','conv_dense_gmacs','deployment_weight_sha256']:
        if not (common[col+'_b1']==common[col+'_b8']).all():raise RuntimeError('Batch architecture/checkpoint mismatch: '+col)
    summary=a.merge(b[['run_id','throughput_b8_images_s','peak_b8_MiB','session_b8']],on='run_id',how='outer',validate='one_to_one')
    # Batch 8 may be completed while batch 1 is still missing; restore identities from catalog.
    ident=performance[keys].set_index('run_id')
    for col in keys[1:]:summary[col]=summary[col].fillna(summary['run_id'].map(ident[col]))
    metric_cols=[c for c in performance.columns if c not in keys]
    summary=summary.merge(performance[['run_id']+metric_cols],on='run_id',how='left',validate='one_to_one')
    summary['profiling_complete']=summary.latency_b1_ms.notna()&summary.throughput_b8_images_s.notna()
    rows=[]
    for (dataset,pipeline,student,batch),group in df.groupby(['dataset','pipeline','student','batch']):
        for col in ['parameters','conv_dense_gmacs','raw_weight_MiB']:
            if group[col].nunique()!=1:raise RuntimeError('Deployment complexity differs within '+str((dataset,pipeline,student,batch))+': '+col)
    for (dataset,pipeline,student),group in summary.groupby(['dataset','pipeline','student']):
        controls=group[group.teacher=='none']
        if len(controls)!=1:continue
        control=controls.iloc[0]
        for _,r in group[group.teacher!='none'].iterrows():
            row={k:r[k] for k in keys};row['control_run_id']=control.run_id
            row['both_profiles_complete']=bool(r.profiling_complete and control.profiling_complete)
            for metric in ['precision','recall','f1','macro_f1','accuracy','roc_auc']:
                row['delta_'+metric]=r[metric]-control[metric]
            row['delta_parameters']=r.parameters-control.parameters
            row['latency_ratio_kd_to_control']=r.latency_b1_ms/control.latency_b1_ms
            row['throughput_ratio_kd_to_control']=r.throughput_b8_images_s/control.throughput_b8_images_s
            row['same_deployment_weights_as_control']=r.deployment_weight_sha256==control.deployment_weight_sha256
            rows.append(row)
    return df,summary,pd.DataFrame(rows)


## Cell 4 — Bekukan kampanye dan daftar pekerjaan
Kampanye ditentukan protokol, versi software, model GPU, sumber, dan kode. Semua sesi tersimpan, sehingga variasi timing antar-sesi tidak disamarkan sebagai variasi seed training.

In [ ]:
SOURCE_CATALOG_SHA256=objsha(CATALOG)
SESSION_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
PROTOCOL=dict(version='b08d_post_kd_efficiency_v1',campaign_label=CAMPAIGN_LABEL,
    input_size=512,precision='float32',tf32=False,xla=False,deterministic=True,
    batches=BATCHES,warmup=WARMUP,trials=TRIALS,iterations=ITERATIONS,
    tensorflow=TF_VERSION,tf_keras=metadata.version('tf-keras'),gpu_info=GPU_INFO,
    builder_sha256=hashlib.sha256(BUILDER_SOURCE.encode()).hexdigest(),
    worker_sha256=hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    orchestration_sha256=ORCHESTRATION_SHA256,catalog_sha256=SOURCE_CATALOG_SHA256,
    verify_saved_test_files=VERIFY_SAVED_TEST_FILES,
    measurement='synthetic GPU-resident input; host forward plus output synchronization; excludes preprocessing and input transfer',
    restore='reviewed checkpoint hashes; strict object restore; CPU export/reload parity; no fresh validation/test inference',
    memory='TensorFlow peak active allocations after warmup, including resident model/input',
    macs='Conv/Dense MACs only; not total FLOPs',training=False,test_inference=False)
CAMPAIGN_ID=objsha(PROTOCOL)[:12]
OUT=RESEARCH/'outputs/benchmark_4datasets/b08d_post_kd_efficiency_v1'/CAMPAIGN_ID
LOCAL=Path('/content/b08d')/CAMPAIGN_ID
OUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
guard(OUT/'protocol.json',PROTOCOL)
BUILDER_PATH=LOCAL/'inference_builder.py';BUILDER_PATH.write_text(BUILDER_SOURCE)
WORKER_PATH=LOCAL/'b08d_worker.py';WORKER_PATH.write_text(WORKER_SOURCE)
guard(OUT/'source_catalog.json',CATALOG)
(OUT/'inference_builder.py').write_text(BUILDER_SOURCE);(OUT/'b08d_worker.py').write_text(WORKER_SOURCE)
RUNS=[r for r in CATALOG if r['dataset'] in DATASETS_TO_RUN and r['pipeline'] in PIPELINES_TO_RUN and r['student'] in STUDENTS_TO_RUN]
save(OUT/'sessions'/f'{SESSION_ID}.json',dict(session_id=SESSION_ID,selected_run_ids=[r['run_id'] for r in RUNS],gpu_info=GPU_INFO,
    note='Timing variance includes different sessions when resumed; timing SD is trial variability, not seed variability.'))
PERFORMANCE=pd.DataFrame([dict(run_id=r['run_id'],dataset=r['dataset'],pipeline=r['pipeline'],student=r['student'],teacher=r['teacher'],training_seed=42,
    **{k:r['metrics'][k] for k in ['precision','recall','f1','macro_f1','accuracy','balanced_accuracy','specificity','roc_auc','average_precision','images','tn','fp','fn','tp','threshold']}) for r in CATALOG])
PERFORMANCE.to_csv(OUT/'saved_test_performance.csv',index=False)
print('Output:',OUT)
print('Selected:',len(RUNS),'checkpoints |',len(RUNS)*len(BATCHES),'GPU jobs. Completed matching jobs will be skipped.')
display(pd.DataFrame(RUNS)[['dataset','pipeline','student','teacher']])


## Cell 5 — Audit sumber sebelum profiling
Memeriksa checkpoint dan metadata terhadap hash laporan B05C/B06C. Hasil test lama diverifikasi secara default. Jika gagal, periksa path/file; tidak perlu training ulang. Cell 8 tetap dapat mengekspor laporan diagnosis.

In [ ]:
PREFLIGHT_PASSED=False
problems=[];rows=[]
for r in RUNS:
    try:
        verify_files(r['metadata_sha256']);verify_files(r['checkpoint_sha256'])
        if VERIFY_SAVED_TEST_FILES:verify_files(r['performance_sha256'])
        folder=relocate(r['checkpoint']).parent
        original_run=r['metrics'].get('run_folder',r['metrics'].get('source_run'))
        runfolder=relocate(original_run);cfg=read(runfolder/'config.json');done=read(runfolder/'COMPLETE.json')
        if cfg['student']!=r['student'] or cfg['teacher']!=r['teacher'] or int(cfg.get('training_seed',cfg.get('seed',-1)))!=42:raise RuntimeError('Run identity mismatch')
        if done['config_sha256']!=sha(runfolder/'config.json'):raise RuntimeError('Run config changed')
        if relocate(done['best_checkpoint'])!=relocate(r['checkpoint']):raise RuntimeError('Selected checkpoint differs from test lock')
        if not bool(r['previous_validation_audit']['passed']):raise RuntimeError('Previous validation audit did not pass')
        rows.append(dict(run_id=r['run_id'],passed=True,checkpoint=r['checkpoint'],previous_validation_max_abs=r['previous_validation_audit']['max_abs_difference']))
        print('Verified',r['run_id'],flush=True)
    except Exception as e:
        problems.append(dict(run_id=r['run_id'],error=str(e)));print('SOURCE ERROR',r['run_id'],str(e),flush=True)
save(OUT/'preflight_errors.json',problems);pd.DataFrame(rows).to_csv(OUT/'preflight_audit.csv',index=False)
if problems:
    raise RuntimeError('Preflight failed. Do not retrain or loosen hash checks. Check paths/files in preflight_errors.json; Cell 8 can export a partial report.')
PREFLIGHT_PASSED=True
print('PASS. Checkpoints and result identities match the reviewed reports. No source files were modified.')


## Cell 6 — Audit ekspor dan profiling
1. Restore hanya backbone dan classifier dengan semua objek model wajib cocok. Entri Teacher/projector/optimizer tidak masuk model deployment.
2. Ekspor dan muat ulang bobot pada CPU: seluruh bobot harus identik, probabilitas input sintetis harus cocok.
3. Proses GPU baru untuk masing-masing batch. Gunakan 30 warm-up, 5 trial × 50 forward calls, float32, TF32/XLA off.

Audit validation pada laporan asal sudah lulus dan diikat melalui hash checkpoint. Notebook ini **tidak mengklaim mengulang audit validation pada citra**. Paritas ekspor memakai input sintetis; ini pemeriksaan serialisasi, bukan validasi performa dataset.

Toleransi audit tidak dilonggarkan otomatis. Jika gagal, jalankan Cell 7–8 untuk mengirim ZIP diagnosis. Skip/resume berlaku hanya untuk pekerjaan dengan marker COMPLETE dan checksum output yang valid.

In [ ]:
if not PREFLIGHT_PASSED:raise RuntimeError('Complete Cell 5 first')
ERRORS=[]
save(OUT/'report_status.json',dict(status='running',selected_checkpoints=len(RUNS),expected_selected_jobs=len(RUNS)*2))
# Shuffle order with a fixed seed to reduce a systematic control-first timing order.
order=np.random.default_rng(2026).permutation(len(RUNS))
for idx in order:
    r=RUNS[int(idx)];rid=r['run_id'];auditdir=OUT/'audits'/rid;auditdir.mkdir(parents=True,exist_ok=True)
    export=LOCAL/'exports'/rid/'deployment.weights.h5';export.parent.mkdir(parents=True,exist_ok=True)
    base=dict(run_id=rid,dataset=r['dataset'],pipeline=r['pipeline'],student=r['student'],teacher=r['teacher'],training_seed=42,
        session_id=SESSION_ID,builder_path=str(BUILDER_PATH),builder_sha256=PROTOCOL['builder_sha256'],export_path=str(export),
        audit_path=str(auditdir/'audit_result.json'),warmup=WARMUP,trials=TRIALS,iterations=ITERATIONS,
        checkpoint_sha256=r['checkpoint_sha256'],worker_sha256=PROTOCOL['worker_sha256'])
    pending=[]
    for batch in BATCHES:
        dest=OUT/'jobs'/rid/f'batch_{batch}';cfg=dict(base,batch=batch,job_dir=str(dest))
        if finished_job(dest,cfg):print('SKIP complete',rid,'batch',batch,flush=True)
        else:pending.append((batch,dest,cfg))
    if not pending:continue
    print('AUDIT',rid,flush=True)
    try:
        # Recheck the exact source before any new profiling job, even after a long pause.
        verify_files(r['checkpoint_sha256'])
        prefix,localhashes=checkpoint_copy(r)
        acfg=dict(base,job_dir=str(auditdir),local_checkpoint=prefix,local_checkpoint_hashes=localhashes)
        ap=auditdir/'config.json';save(ap,acfg)
        run_process('audit',ap,auditdir/'worker.log')
        a=read(auditdir/'audit_result.json')
        if a['status']!='complete' or a['configuration_sha256']!=sha(ap):raise RuntimeError('Export audit incomplete')
    except Exception as e:
        ERRORS.append(dict(run_id=rid,stage='audit',error=str(e)));save(OUT/'errors_latest_attempt.json',ERRORS)
        print('FAILED audit',rid,str(e)[-1600:],flush=True);continue
    for batch,dest,cfg in pending:
        dest.mkdir(parents=True,exist_ok=True);cp=dest/'config.json'
        if cp.exists() and not config_matches(read(cp),cfg):raise RuntimeError('Refusing incompatible job overwrite: '+str(dest))
        save(cp,cfg)
        # A failed earlier result must never be accepted as completion for this attempt.
        if (dest/'result.json').exists():(dest/'result.json').unlink()
        print('PROFILE',rid,'batch',batch,flush=True)
        try:
            run_process('profile',cp,dest/'worker.log');rr=read(dest/'result.json')
            if rr['status']!='complete' or rr['configuration_sha256']!=sha(cp):raise RuntimeError('Result/config mismatch')
            outputs={p.name:sha(p) for p in dest.iterdir() if p.is_file() and p.name!='COMPLETE.json'}
            save(dest/'COMPLETE.json',dict(outputs=outputs,status='complete'))
        except Exception as e:
            ERRORS.append(dict(run_id=rid,batch=batch,stage='profiling',error=str(e)))
            save(OUT/'errors_latest_attempt.json',ERRORS);print('FAILED',rid,batch,str(e)[-1600:],flush=True)
    # Large local weights need not be retained once this run is done. Drive sources are untouched.
    if export.exists():export.unlink()
    shutil.rmtree(LOCAL/'checkpoints'/rid,ignore_errors=True)
save(OUT/'errors_latest_attempt.json',ERRORS)
print('Profiling loop finished. Continue to Cells 7 and 8, including after partial failures.')


## Cell 7 — Gabungkan hasil dan bandingkan dengan kontrol
B1 = batch 1; B8 = batch 8. ΔF1 = KD − kontrol dari pipeline/dataset/Student yang sama. Rasio latensi KD/kontrol < 1 hanya perbedaan timing deskriptif; bukan bukti KD mempercepat model. Profil belum lengkap tetap ditandai partial.

In [ ]:
RESULTS=[];INTEGRITY_ERRORS=[]
for result_path in sorted((OUT/'jobs').glob('*/batch_*/result.json')):
    job=result_path.parent
    try:
        if not (job/'COMPLETE.json').exists():continue
        cfg=read(job/'config.json')
        if finished_job(job,cfg):RESULTS.append(read(result_path))
    except Exception as e:INTEGRITY_ERRORS.append(dict(path=str(job),error=str(e)))
ALL_BATCH,SUMMARY,DELTAS=summary_tables(RESULTS,PERFORMANCE)
for name,table in [('all_batch_results',ALL_BATCH),('performance_efficiency',SUMMARY),('kd_vs_own_control',DELTAS)]:
    table.to_csv(OUT/(name+'.csv'),index=False)
complete_ids=set(SUMMARY.loc[SUMMARY.profiling_complete,'run_id']) if len(SUMMARY) else set()
selected_ids={r['run_id'] for r in RUNS}
STATUS=dict(status='complete_selected' if selected_ids<=complete_ids and not INTEGRITY_ERRORS else 'partial',
    completed_gpu_jobs=len(RESULTS),completed_checkpoint_profiles=len(complete_ids),
    selected_checkpoint_profiles=len(selected_ids),all_catalog_checkpoint_profiles=48,
    complete_all_48=len(complete_ids)==48 and not INTEGRITY_ERRORS,
    missing_selected_run_ids=sorted(selected_ids-complete_ids),integrity_errors=INTEGRITY_ERRORS,
    note='Each complete checkpoint profile requires batch 1 and 8. Metrics come from archived tests; no new test inference.')
save(OUT/'report_status.json',STATUS)
print(STATUS)
if len(SUMMARY):
    display(SUMMARY[['dataset','pipeline','student','teacher','f1','parameters','conv_dense_gmacs','latency_b1_ms','latency_b1_trial_sd_ms','throughput_b8_images_s','peak_b1_MiB','peak_b8_MiB','profiling_complete']].round(4))
    if len(DELTAS):display(DELTAS[['dataset','pipeline','student','teacher','delta_f1','delta_parameters','latency_ratio_kd_to_control','same_deployment_weights_as_control','both_profiles_complete']].round(4))
else:print('No completed GPU results yet. Export the partial report in Cell 8.')


## Cell 8 — Excel dan ZIP
ZIP berisi hasil, timing per panggilan, audit, manifest, log, dan kode. Tidak menyertakan checkpoint besar atau citra. Bisa dijalankan meskipun preflight atau worker gagal setelah Cell 4.

In [ ]:
# Can also be run after a failed preflight/worker to share diagnostic files.
notes=[
 ('Scope','Student controls and post-KD checkpoints; 4 datasets x 2 pipelines x 2 students x 3 teacher conditions = 48.'),
 ('No retraining','Only checkpoint restore, CPU export audit and GPU forward profiling. No test/validation images accessed.'),
 ('Identity','Checkpoint hashes and saved performance are bound to the reviewed B05C/B06C reports embedded in this notebook.'),
 ('Restore audit','Strict backbone/classifier object matching; CPU export/reload checks identical weights and synthetic predictions. Previous validation audit is retained, not rerun.'),
 ('Control','Compare each KD run to the control from its own dataset, student and pipeline.'),
 ('Parameters','Deployment parameter count excludes teacher, projector and optimizer. Training checkpoint file size is not the deployment size.'),
 ('MACs','Conv/Dense GMACs only. Not total FLOPs; BN, pooling, activation and addition are excluded.'),
 ('Latency','Batch 1 forward + host/output synchronization. Excludes decode/resize/preprocessing and host-to-device transfer.'),
 ('Throughput','Batch 8 images per second; not end-to-end clinical throughput.'),
 ('Memory','Peak active TensorFlow GPU allocations after warmup including resident model/input; not total reserved GPU memory.'),
 ('SD','Sample SD across five trial means; not variability across classifier seeds.'),
 ('Timing interpretation','Small timing differences are descriptive noise unless investigated. No equivalence/significance test is performed.'),
 ('Inference vs training','No teacher-cache cost, training overhead or energy consumption is measured.'),
 ('Resume','Completed jobs are hash-verified. Same hardware model/software campaign can span sessions; session IDs are retained.'),
 ('Scope of claims','Same student architecture can preserve inference cost after KD; benefit in classification depends on dataset and pairing.'),
 ('Existing test limitation','Test partitions were previously inspected. This profiling does not make them untouched confirmatory sets.'),
 ('TensorFlow checkpoint API','https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint'),
 ('TensorFlow memory API','https://www.tensorflow.org/api_docs/python/tf/config/experimental/reset_memory_stats')]
frames={'Method_notes':pd.DataFrame(notes,columns=['Item','Explanation']),
        'Saved_test_metrics':PERFORMANCE,
        'Current_selection':pd.DataFrame(RUNS)[['run_id','dataset','pipeline','student','teacher','checkpoint','report']]}
for name,filename in [('Performance_efficiency','performance_efficiency.csv'),('All_batches','all_batch_results.csv'),('KD_vs_own_control','kd_vs_own_control.csv'),('Source_audit','preflight_audit.csv')]:
    p=OUT/filename
    if p.exists():
        try:frames[name]=pd.read_csv(p)
        except pd.errors.EmptyDataError:pass
frames['Status']=pd.DataFrame([{'item':k,'value':json.dumps(v) if isinstance(v,(list,dict)) else v} for k,v in read(OUT/'report_status.json').items()]) if (OUT/'report_status.json').exists() else pd.DataFrame([{'item':'status','value':'preflight only; no profiling yet'}])
from openpyxl.styles import Font,PatternFill,Alignment
xlsx=OUT/'B08D_post_KD_efficiency.xlsx'
with pd.ExcelWriter(xlsx,engine='openpyxl') as writer:
    for name,df in frames.items():
        df.to_excel(writer,sheet_name=name,index=False);ws=writer.sheets[name];ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
        for cell in ws[1]:cell.font=Font(bold=True,color='FFFFFF');cell.fill=PatternFill('solid',fgColor='17466B')
        for column in ws.columns:
            letter=column[0].column_letter;ws.column_dimensions[letter].width=min(55,max(13,max(len(str(c.value or '')) for c in column[:100])+2))
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value,float):cell.number_format='0.0000'
                # Treat all source strings as text, not executable Excel formulas.
                if isinstance(cell.value,str) and cell.value.startswith(('=','+','-','@')):cell.data_type='s'
    writer.sheets['Method_notes'].column_dimensions['B'].width=105
    for row in writer.sheets['Method_notes'].iter_rows(min_row=2):
        row[1].alignment=Alignment(wrap_text=True,vertical='top');writer.sheets['Method_notes'].row_dimensions[row[0].row].height=45
REPORT=OUT/f'B08D_post_KD_efficiency_{CAMPAIGN_ID}_report.zip'
with zipfile.ZipFile(REPORT,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix.lower() not in ['.zip','.h5'] and '.data-' not in p.name:
            z.write(p,p.relative_to(OUT))
print('Excel:',xlsx);print('Report:',REPORT)
if DOWNLOAD_REPORT:files.download(str(REPORT))


## Membaca hasil dengan benar
- Parameter deployment dan GMACs seharusnya sama pada arsitektur Student yang sama; hal ini diperiksa dari model yang benar-benar dimuat, bukan disalin dari tabel B08C.
- `same_deployment_weights_as_control=True` berarti bobot terbaik memang identik; peningkatan feature loss tidak otomatis menjamin checkpoint akhir berubah. Nilai ini hanya audit, bukan error.
- Parameter sama tidak membuktikan latensi identik. Laporkan pengukuran dan SD trial tanpa menyatakan equivalence/significance.
- Perbandingan performa menggunakan hasil test lama yang sebelumnya sudah diperiksa; keterbatasan reused holdout tetap berlaku.
- Ukuran checkpoint training dapat mencakup proyektor/optimizer. Kolom ukuran bobot di sini adalah bobot **deployment** yang diekspor.
- Memori adalah alokasi aktif TensorFlow setelah pemanasan, bukan total VRAM reserved. Timing memakai input sintetis yang sudah berada di GPU, bukan end-to-end pemrosesan mammogram.
- GMACs hanya Conv/Dense. Tidak diberi label total FLOPs. Biaya training, cache Teacher, dan energi tidak diukur.
- Profil Teacher lama B08C tidak dicampur ke pengukuran baru ini. Paper dapat menyajikannya sebagai eksperimen terpisah dengan keterangan sesi/protokol.

Sumber API: [TensorFlow Checkpoint](https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint), [reset_memory_stats](https://www.tensorflow.org/api_docs/python/tf/config/experimental/reset_memory_stats).
